# D1 Surface Classifier v5.0 — Эксперименты

## Обзор пайплайна

**Изменения таксономии v4.0:**
- `negative_feedback` → `feedback` (L1)
- 20 → 24 L2 класса (+4 подтипа положительной обратной связи)
- Новые пороги confidence: 0.90 / 0.70 / 0.50

**Датасет:** v5 (12 000 сэмплов, 24 L2 класса)

**Методы (legacy, до аудита):**
1. TF-IDF + Logistic Regression (L1/L2 отдельно)
2. GridSearchCV TF-IDF + LinearSVC (L1/L2 отдельно)
3. EmbeddingClassifier (rubert-tiny2, только L1)
4. SetFit (n_samples_per_class=32)
5. Cascade Classifier (Rule → ML calibrated → LLM)
6. Multi-Intent Detection (экспериментальный)

**Метрики:**
- F1 Macro (L1/L2), Accuracy
- negative_recall, positive_recall, feedback_f1
- LLM Reduction %

> ⚠️ **Примечание:** Ячейки 1-30 — результаты до аудита (legacy). Секция v5.0 (ячейки 31+) содержит исправленную методологию с statistical rigor.

In [1]:
!pip install setfit 

In [2]:
# =============================================================================
# Ячейка 1: Настройка и импорты
# =============================================================================
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import json
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, recall_score, precision_score
)
from sklearn.calibration import CalibratedClassifierCV
import joblib

# Путь к проекту
sys.path.insert(0, str(Path('.').resolve()))

# Импорт таксономии и утилит
from utils.taxonomy import (
    INTENT_LABELS_L1, INTENT_LABELS_L2,
    L2_TO_L1, CLASS_WEIGHTS_L1, CLASS_WEIGHTS_L2,
    CONFIDENCE_THRESHOLDS, get_sklearn_class_weights,
    get_taxonomy_stats
)
from utils.classifiers import (
    RuleBasedClassifier, EmbeddingClassifier, CascadeClassifier
)

# Воспроизводимость
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Директории для результатов
OUTPUT_DIR = Path('outputs')
OUTPUT_MODELS = OUTPUT_DIR / 'models'
OUTPUT_FIGURES = OUTPUT_DIR / 'figures'
OUTPUT_TABLES = OUTPUT_DIR / 'tables'
OUTPUT_REPORTS = OUTPUT_DIR / 'reports'

for d in [OUTPUT_MODELS, OUTPUT_FIGURES, OUTPUT_TABLES, OUTPUT_REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Версия таксономии: {get_taxonomy_stats()["version"]}')
print(f'L1 классов: {len(INTENT_LABELS_L1)}')
print(f'L2 классов: {len(INTENT_LABELS_L2)}')
print(f'Пороги confidence: {CONFIDENCE_THRESHOLDS}')

Версия таксономии: 4.0
L1 классов: 5
L2 классов: 24
Пороги confidence: {'high': 0.9, 'medium': 0.7, 'low': 0.5}


In [3]:
# =============================================================================
# Ячейка 2: Загрузка датасета v5
# =============================================================================
DATA_DIR = Path('data')

df_full = pd.read_csv(DATA_DIR / 'd1_messages_v5_full.csv')
df_train = pd.read_csv(DATA_DIR / 'd1_messages_v5_train.csv')
df_val = pd.read_csv(DATA_DIR / 'd1_messages_v5_val.csv')
df_test = pd.read_csv(DATA_DIR / 'd1_messages_v5_test.csv')

print(f'Датасет v5 загружен:')
print(f'  Полный:      {len(df_full):,} сэмплов')
print(f'  Train:       {len(df_train):,} сэмплов')
print(f'  Validation:  {len(df_val):,} сэмплов')
print(f'  Test:        {len(df_test):,} сэмплов')

# Извлечение признаков и меток
X_train = df_train['text'].values
y_train_l1 = df_train['label_l1'].values
y_train_l2 = df_train['label_l2'].values

X_val = df_val['text'].values
y_val_l1 = df_val['label_l1'].values
y_val_l2 = df_val['label_l2'].values

X_test = df_test['text'].values
y_test_l1 = df_test['label_l1'].values
y_test_l2 = df_test['label_l2'].values

print(f'\nРаспределение L1 (train):')
print(df_train['label_l1'].value_counts())

Датасет v5 загружен:
  Полный:      12,000 сэмплов
  Train:       8,399 сэмплов
  Validation:  1,801 сэмплов
  Test:        1,800 сэмплов

Распределение L1 (train):
label_l1
feedback          2800
faq               1750
conversational    1749
booking           1050
anamnesis         1050
Name: count, dtype: int64


In [4]:
# =============================================================================
# Ячейка 3: EDA — анализ классов обратной связи (Feedback)
# =============================================================================
print('=== Анализ классов Feedback ===')

# Фильтруем сэмплы feedback
df_feedback = df_full[df_full['label_l1'] == 'feedback']
print(f'Всего сэмплов feedback: {len(df_feedback)}')

# Распределение negative vs positive
df_feedback['sentiment'] = df_feedback['label_l2'].apply(
    lambda x: 'negative' if x.startswith('negative_') else 'positive'
)
print(f'\nРаспределение по тональности:')
print(df_feedback['sentiment'].value_counts())

# Распределение L2 внутри feedback
print(f'\nРаспределение L2 внутри feedback:')
print(df_feedback['label_l2'].value_counts())

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Распределение L1
df_full['label_l1'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Распределение L1 классов')
axes[0].set_xlabel('L1 класс')
axes[0].set_ylabel('Количество')
axes[0].tick_params(axis='x', rotation=45)

# Тональность feedback
df_feedback['sentiment'].value_counts().plot(kind='bar', ax=axes[1], color=['red', 'green'])
axes[1].set_title('Тональность Feedback')
axes[1].set_xlabel('Тональность')
axes[1].set_ylabel('Количество')

plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'v5_dataset_distribution.png', dpi=150)
plt.show()

=== Анализ классов Feedback ===
Всего сэмплов feedback: 4000

Распределение по тональности:
sentiment
negative    2000
positive    2000
Name: count, dtype: int64

Распределение L2 внутри feedback:
label_l2
negative_general    500
positive_service    500
positive_general    500
negative_staff      500
negative_quality    500
negative_service    500
positive_staff      500
positive_quality    500
Name: count, dtype: int64


## Метод 1: TF-IDF + Logistic Regression (legacy, L1/L2 отдельно)

In [5]:
# =============================================================================
# Ячейка 4: TF-IDF векторизация
# =============================================================================
print('=== TF-IDF векторизация ===')

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print(f'Размер словаря TF-IDF: {len(tfidf.vocabulary_)}')
print(f'Размерность train: {X_train_tfidf.shape}')

=== TF-IDF векторизация ===
Размер словаря TF-IDF: 3189
Размерность train: (8399, 3189)


In [6]:
# =============================================================================
# Ячейка 5: Logistic Regression — L1 (отдельная модель)
# =============================================================================
print('=== Logistic Regression (L1) ===')

# Веса классов
class_weights_l1 = get_sklearn_class_weights('l1')

lr_l1 = LogisticRegression(
    max_iter=1000,
    class_weight=class_weights_l1,
    random_state=SEED,
    n_jobs=-1,
)
lr_l1.fit(X_train_tfidf, y_train_l1)

# Оценка на validation
y_pred_l1 = lr_l1.predict(X_val_tfidf)

acc_l1 = accuracy_score(y_val_l1, y_pred_l1)
f1_macro_l1 = f1_score(y_val_l1, y_pred_l1, average='macro')
f1_weighted_l1 = f1_score(y_val_l1, y_pred_l1, average='weighted')

print(f'Результаты L1 (validation):')
print(f'  Accuracy: {acc_l1:.4f}')
print(f'  F1 Macro: {f1_macro_l1:.4f}')
print(f'  F1 Weighted: {f1_weighted_l1:.4f}')

print(f'\nОтчёт классификации (L1):')
print(classification_report(y_val_l1, y_pred_l1))

=== Logistic Regression (L1) ===
Результаты L1 (validation):
  Accuracy: 0.9634
  F1 Macro: 0.9654
  F1 Weighted: 0.9635

Отчёт классификации (L1):
                precision    recall  f1-score   support

     anamnesis       0.98      0.95      0.97       225
       booking       0.99      0.96      0.97       225
conversational       0.99      0.93      0.96       376
           faq       0.99      0.96      0.97       375
      feedback       0.92      0.99      0.96       600

      accuracy                           0.96      1801
     macro avg       0.97      0.96      0.97      1801
  weighted avg       0.96      0.96      0.96      1801



In [7]:
# =============================================================================
# Ячейка 6: Logistic Regression — L2 (отдельная модель)
# =============================================================================
print('=== Logistic Regression (L2) ===')

# Веса классов L2
class_weights_l2 = get_sklearn_class_weights('l2')

lr_l2 = LogisticRegression(
    max_iter=1000,
    class_weight=class_weights_l2,
    random_state=SEED,
    n_jobs=-1,
)
lr_l2.fit(X_train_tfidf, y_train_l2)

# Оценка на validation
y_pred_l2 = lr_l2.predict(X_val_tfidf)

acc_l2 = accuracy_score(y_val_l2, y_pred_l2)
f1_macro_l2 = f1_score(y_val_l2, y_pred_l2, average='macro')
f1_weighted_l2 = f1_score(y_val_l2, y_pred_l2, average='weighted')

print(f'Результаты L2 (validation):')
print(f'  Accuracy: {acc_l2:.4f}')
print(f'  F1 Macro: {f1_macro_l2:.4f}')
print(f'  F1 Weighted: {f1_weighted_l2:.4f}')

=== Logistic Regression (L2) ===
Результаты L2 (validation):
  Accuracy: 0.9511
  F1 Macro: 0.9526
  F1 Weighted: 0.9525


## Метод 2: GridSearchCV TF-IDF + LinearSVC (legacy, L1/L2 отдельно)

In [8]:
# =============================================================================
# Ячейка 7: GridSearchCV для L1
# =============================================================================
print('=== GridSearchCV (L1) ===')

param_grid = {
    'C': [0.1, 1.0, 10.0],
    'max_iter': [1000, 2000],
}

svc_l1 = LinearSVC(class_weight=class_weights_l1, random_state=SEED)

grid_search_l1 = GridSearchCV(
    svc_l1, param_grid, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search_l1.fit(X_train_tfidf, y_train_l1)

print(f'Лучшие параметры (L1): {grid_search_l1.best_params_}')
print(f'Лучший F1 Macro (CV): {grid_search_l1.best_score_:.4f}')

# Оценка лучшей модели
best_svc_l1 = grid_search_l1.best_estimator_
y_pred_svc_l1 = best_svc_l1.predict(X_val_tfidf)

print(f'\nF1 Macro (validation): {f1_score(y_val_l1, y_pred_svc_l1, average="macro"):.4f}')

=== GridSearchCV (L1) ===
Fitting 3 folds for each of 6 candidates, totalling 18 fits


/opt/anaconda3/envs/ml-python312/lib/python3.12/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/opt/anaconda3/envs/ml-python312/lib/python3.12/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/opt/anaconda3/envs/ml-python312/lib/python3.12/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/opt/anaconda3/envs/ml-python312/lib/python3.12/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
 

Лучшие параметры (L1): {'C': 1.0, 'max_iter': 1000}
Лучший F1 Macro (CV): 0.9772

F1 Macro (validation): 0.9824


In [9]:
# =============================================================================
# Ячейка 8: GridSearchCV для L2
# =============================================================================
print('=== GridSearchCV (L2) ===')

svc_l2 = LinearSVC(class_weight=class_weights_l2, random_state=SEED)

grid_search_l2 = GridSearchCV(
    svc_l2, param_grid, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search_l2.fit(X_train_tfidf, y_train_l2)

print(f'Лучшие параметры (L2): {grid_search_l2.best_params_}')
print(f'Лучший F1 Macro (CV): {grid_search_l2.best_score_:.4f}')

# Оценка лучшей модели
best_svc_l2 = grid_search_l2.best_estimator_
y_pred_svc_l2 = best_svc_l2.predict(X_val_tfidf)

print(f'\nF1 Macro (validation): {f1_score(y_val_l2, y_pred_svc_l2, average="macro"):.4f}')

=== GridSearchCV (L2) ===
Fitting 3 folds for each of 6 candidates, totalling 18 fits


/opt/anaconda3/envs/ml-python312/lib/python3.12/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/opt/anaconda3/envs/ml-python312/lib/python3.12/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/opt/anaconda3/envs/ml-python312/lib/python3.12/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/opt/anaconda3/envs/ml-python312/lib/python3.12/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
 

Лучшие параметры (L2): {'C': 1.0, 'max_iter': 1000}
Лучший F1 Macro (CV): 0.9648

F1 Macro (validation): 0.9600


## Метод 3: Sentence-Transformers (Embedding Classifier, только L1)

In [10]:
# =============================================================================
# Ячейка 9: Проверка доступности Sentence-Transformers
# =============================================================================
HAS_SBERT = False
try:
    from sentence_transformers import SentenceTransformer
    HAS_SBERT = True
    print('sentence-transformers доступен')
except ImportError:
    print('sentence-transformers не установлен. Запустите: pip install sentence-transformers')

sentence-transformers доступен


In [11]:
# =============================================================================
# Ячейка 10: EmbeddingClassifier — основная модель (rubert-tiny2), только L1
# =============================================================================
if HAS_SBERT:
    print('=== EmbeddingClassifier (rubert-tiny2) — только L1 ===')
    
    # Основная модель: rubert-tiny2 (быстрая)
    emb_clf_l1_primary = EmbeddingClassifier(
        model_name='cointegrated/rubert-tiny2',
    )
    
    print('Обучение L1 классификатора...')
    emb_clf_l1_primary.fit(X_train.tolist(), y_train_l1.tolist())
    
    # Оценка
    results = emb_clf_l1_primary.predict_with_confidence(X_val.tolist())
    y_pred_emb_l1 = [r[0] for r in results]
    
    emb_acc_l1 = accuracy_score(y_val_l1, y_pred_emb_l1)
    emb_f1_l1 = f1_score(y_val_l1, y_pred_emb_l1, average='macro')
    
    print(f'\nОсновная модель (rubert-tiny2) — результаты L1:')
    print(f'  Accuracy: {emb_acc_l1:.4f}')
    print(f'  F1 Macro: {emb_f1_l1:.4f}')
else:
    print('EmbeddingClassifier пропущен (sentence-transformers недоступен)')

=== EmbeddingClassifier (rubert-tiny2) — только L1 ===
Обучение L1 классификатора...


Batches:   0%|          | 0/263 [00:00<?, ?it/s]


Основная модель (rubert-tiny2) — результаты L1:
  Accuracy: 0.9389
  F1 Macro: 0.9368


In [12]:
# =============================================================================
# Ячейка 11: EmbeddingClassifier — вторичная модель (ru-en-RoSBERTa)
# ЗАКОММЕНТИРОВАНО: долгое обучение, не используется в финальном сравнении
# =============================================================================
# if HAS_SBERT:
#     print('=== EmbeddingClassifier (ru-en-RoSBERTa) — вторичная модель ===')
#     try:
#         emb_clf_l1_secondary = EmbeddingClassifier(
#             model_name='ai-forever/ru-en-RoSBERTa',
#         )
#         print('Обучение L1 классификатора (вторичная модель)...')
#         emb_clf_l1_secondary.fit(X_train.tolist(), y_train_l1.tolist())
#         results = emb_clf_l1_secondary.predict_with_confidence(X_val.tolist())
#         y_pred_emb_l1_sec = [r[0] for r in results]
#         emb_acc_l1_sec = accuracy_score(y_val_l1, y_pred_emb_l1_sec)
#         emb_f1_l1_sec = f1_score(y_val_l1, y_pred_emb_l1_sec, average='macro')
#         print(f'\nВторичная модель (ru-en-RoSBERTa) — результаты L1:')
#         print(f'  Accuracy: {emb_acc_l1_sec:.4f}')
#         print(f'  F1 Macro: {emb_f1_l1_sec:.4f}')
#     except Exception as e:
#         print(f'ru-en-RoSBERTa ошибка: {e}')
#         print('Используем rubert-tiny2 как fallback')

## Метод 4: SetFit (Few-Shot обучение)

In [13]:
# =============================================================================
# Ячейка 12: Проверка доступности SetFit
# =============================================================================
HAS_SETFIT = False
try:
    from setfit import SetFitModel, Trainer, TrainingArguments
    from datasets import Dataset
    HAS_SETFIT = True
    print('SetFit доступен')
except ImportError:
    print('SetFit не установлен. Запустите: pip install setfit')

SetFit не установлен. Запустите: pip install setfit


In [14]:
# =============================================================================
# Ячейка 13: Обучение SetFit (few-shot, 32 сэмпла на класс)
# =============================================================================
if HAS_SETFIT:
    print('=== SetFit (n_samples=32 на класс) ===')
    
    # Выборка few-shot данных
    n_samples_per_class = 32  # v4.0: увеличено с 16
    
    few_shot_data = []
    for l1 in INTENT_LABELS_L1:
        l1_samples = df_train[df_train['label_l1'] == l1].sample(
            min(n_samples_per_class, len(df_train[df_train['label_l1'] == l1])),
            random_state=SEED
        )
        few_shot_data.append(l1_samples)
    
    df_few_shot = pd.concat(few_shot_data)
    print(f'Few-shot сэмплов: {len(df_few_shot)}')
    
    # Создание датасета
    train_dataset = Dataset.from_dict({
        'text': df_few_shot['text'].tolist(),
        'label': df_few_shot['label_l1'].tolist(),
    })
    
    # Инициализация SetFit
    setfit_model = SetFitModel.from_pretrained('cointegrated/rubert-tiny2')
    
    # Обучение
    trainer = Trainer(
        model=setfit_model,
        train_dataset=train_dataset,
        args=TrainingArguments(
            num_epochs=1,
            batch_size=16,
        ),
    )
    trainer.train()
    
    # Оценка
    y_pred_setfit = setfit_model.predict(X_val.tolist())
    
    setfit_acc = accuracy_score(y_val_l1, y_pred_setfit)
    setfit_f1 = f1_score(y_val_l1, y_pred_setfit, average='macro')
    
    print(f'\nРезультаты SetFit:')
    print(f'  Accuracy: {setfit_acc:.4f}')
    print(f'  F1 Macro: {setfit_f1:.4f}')
else:
    print('SetFit пропущен')

SetFit пропущен


## Метод 5: Каскадный классификатор (Правила → ML → LLM)

In [15]:
# =============================================================================
# Ячейка 14: Классификатор на правилах (Rule-Based)
# =============================================================================
print('=== Классификатор на правилах ===')

rule_clf = RuleBasedClassifier()

# Оценка покрытия
rule_matches = 0
rule_correct = 0

for text, true_l1 in zip(X_val[:500], y_val_l1[:500]):
    intent, conf, matched = rule_clf.classify(text)
    if matched:
        rule_matches += 1
        # Маппинг правил на L1
        intent_map = {
            'booking': 'booking',
            'reschedule_cancel': 'booking',
            'complaint_primary': 'anamnesis',
            'price_question': 'faq',
            'clinic_faq': 'faq',
            'visit_recommendations': 'faq',
            'followup_question': 'faq',
            'negative_feedback': 'feedback',
            'positive_feedback': 'feedback',
            'other': 'conversational',
        }
        pred_l1 = intent_map.get(intent, 'faq')
        if pred_l1 == true_l1:
            rule_correct += 1

coverage = rule_matches / 500 * 100
precision = rule_correct / rule_matches * 100 if rule_matches > 0 else 0

print(f'Результаты Rule-Based (500 сэмплов):')
print(f'  Покрытие: {coverage:.1f}%')
print(f'  Точность: {precision:.1f}%')
print(f'  Статистика: {rule_clf.get_stats()}')

=== Классификатор на правилах ===
Результаты Rule-Based (500 сэмплов):
  Покрытие: 43.6%
  Точность: 21.6%
  Статистика: {'total_calls': 500, 'matched': 218, 'not_matched': 282, 'by_intent': {'negative_general': 50, 'clinic_info': 13, 'visit_prep': 7, 'cancel': 14, 'gratitude': 7, 'symptom': 27, 'new_appointment': 31, 'reschedule': 17, 'positive_general': 13, 'price': 15, 'procedure': 14, 'farewell': 9, 'unclear': 1}, 'coverage': 0.436}


In [16]:
# =============================================================================
# Ячейка 15: Каскадный классификатор (Rule → ML → LLM)
# =============================================================================
print('=== Каскадный классификатор (Правила → ML → LLM) ===')

if HAS_SBERT and 'emb_clf_l1_primary' in dir():
    # Создание каскада с калибровкой
    cascade_clf = CascadeClassifier(
        rule_classifier=rule_clf,
        ml_classifier=emb_clf_l1_primary,
        llm_client=None,  # LLM не используется в этом эксперименте
        ml_threshold=0.85,
        llm_threshold=0.50,
        calibrate=True,
    )
    
    # Оценка на выборке
    CASCADE_SAMPLE = min(300, len(X_val))
    cascade_preds = []
    cascade_sources = {'rule': 0, 'ml': 0, 'llm': 0}
    
    for text in tqdm(X_val[:CASCADE_SAMPLE], desc='Каскад'):
        intent, source, conf = cascade_clf.classify(text)
        cascade_preds.append(intent)
        cascade_sources[source] += 1
    
    cascade_acc = accuracy_score(y_val_l1[:CASCADE_SAMPLE], cascade_preds)
    cascade_f1 = f1_score(y_val_l1[:CASCADE_SAMPLE], cascade_preds, average='macro')
    
    print(f'\nРезультаты каскада ({CASCADE_SAMPLE} сэмплов):')
    print(f'  Accuracy: {cascade_acc:.4f}')
    print(f'  F1 Macro: {cascade_f1:.4f}')
    print(f'\nРаспределение по слоям:')
    for layer, count in cascade_sources.items():
        pct = count / CASCADE_SAMPLE * 100
        print(f'  {layer}: {count} ({pct:.1f}%)')
    
    llm_reduction = (1 - cascade_sources['llm'] / CASCADE_SAMPLE) * 100
    print(f'\nСнижение LLM вызовов: {llm_reduction:.1f}%')
else:
    print('Каскад пропущен (EmbeddingClassifier недоступен)')

=== Каскадный классификатор (Правила → ML → LLM) ===


Каскад: 100%|██████████| 300/300 [00:00<00:00, 317.96it/s]


Результаты каскада (300 сэмплов):
  Accuracy: 0.5533
  F1 Macro: 0.1910

Распределение по слоям:
  rule: 128 (42.7%)
  ml: 172 (57.3%)
  llm: 0 (0.0%)

Снижение LLM вызовов: 100.0%


## Метрики для Feedback-классов

In [17]:
# =============================================================================
# Ячейка 16: Метрики для Feedback-классов
# =============================================================================
print('=== Метрики Feedback ===')

# Предсказания лучшей модели (GridSearchCV L2)
y_pred_best = best_svc_l2.predict(X_val_tfidf)

# Фильтруем сэмплы feedback
feedback_mask = df_val['label_l1'] == 'feedback'
y_true_feedback = y_val_l2[feedback_mask]
y_pred_feedback = y_pred_best[feedback_mask]

# Полнота негативных отзывов
neg_classes = ['negative_service', 'negative_quality', 'negative_staff', 'negative_general']
neg_mask = np.isin(y_true_feedback, neg_classes)

if neg_mask.sum() > 0:
    neg_recall = recall_score(
        y_true_feedback[neg_mask], y_pred_feedback[neg_mask],
        average='micro', zero_division=0
    )
    print(f'Полнота негативных отзывов: {neg_recall:.4f}')

# Полнота позитивных отзывов
pos_classes = ['positive_service', 'positive_quality', 'positive_staff', 'positive_general']
pos_mask = np.isin(y_true_feedback, pos_classes)

if pos_mask.sum() > 0:
    pos_recall = recall_score(
        y_true_feedback[pos_mask], y_pred_feedback[pos_mask],
        average='micro', zero_division=0
    )
    print(f'Полнота позитивных отзывов: {pos_recall:.4f}')

# F1 для feedback в целом
feedback_f1 = f1_score(y_true_feedback, y_pred_feedback, average='macro', zero_division=0)
print(f'F1 Macro (feedback): {feedback_f1:.4f}')

=== Метрики Feedback ===
Полнота негативных отзывов: 0.9700
Полнота позитивных отзывов: 0.9833
F1 Macro (feedback): 0.6060


## Детекция мульти-интентов (экспериментальный)

In [18]:
# =============================================================================
# Ячейка 17: Детекция мульти-интентов (экспериментальный)
# =============================================================================
print('=== Детекция мульти-интентов (экспериментальный) ===')

def detect_multi_intent(text, classifier, threshold=0.3):
    """Определение нескольких интентов в одном сообщении."""
    try:
        probs = classifier.predict_proba(tfidf.transform([text]))[0]
        intents = []
        for label, prob in zip(classifier.classes_, probs):
            if prob >= threshold:
                intents.append({'intent': label, 'confidence': prob})
        return sorted(intents, key=lambda x: -x['confidence'])
    except:
        return []

# Тестовые примеры
MULTI_INTENT_SAMPLES = [
    "Зуб болит и хочу записаться",           # anamnesis + booking
    "Спасибо за лечение, теперь нужна чистка", # feedback + booking
    "Врач был груб, запишите к другому",       # feedback + booking
    "Сколько стоит и когда можно прийти",      # faq + booking
]

print('Результаты теста мульти-интентов:')
for sample in MULTI_INTENT_SAMPLES:
    intents = detect_multi_intent(sample, lr_l1, threshold=0.25)
    print(f'\n"{sample}"')
    for intent in intents[:3]:
        print(f'  - {intent["intent"]}: {intent["confidence"]:.3f}')

=== Детекция мульти-интентов (экспериментальный) ===
Результаты теста мульти-интентов:

"Зуб болит и хочу записаться"
  - anamnesis: 0.975

"Спасибо за лечение, теперь нужна чистка"
  - feedback: 0.378

"Врач был груб, запишите к другому"
  - feedback: 0.947

"Сколько стоит и когда можно прийти"
  - faq: 0.855


## Сравнение результатов (legacy v4.0)

In [19]:
# =============================================================================
# Ячейка 18: Сводка результатов (legacy)
# =============================================================================
print('=== Сводка результатов (legacy) ===')

results = {
    'TF-IDF + LR (L1)': {'accuracy': acc_l1, 'f1_macro': f1_macro_l1},
    'TF-IDF + LR (L2)': {'accuracy': acc_l2, 'f1_macro': f1_macro_l2},
    'GridSearchCV SVC (L1)': {
        'accuracy': accuracy_score(y_val_l1, y_pred_svc_l1),
        'f1_macro': f1_score(y_val_l1, y_pred_svc_l1, average='macro')
    },
    'GridSearchCV SVC (L2)': {
        'accuracy': accuracy_score(y_val_l2, y_pred_svc_l2),
        'f1_macro': f1_score(y_val_l2, y_pred_svc_l2, average='macro')
    },
}

if HAS_SBERT and 'emb_acc_l1' in dir():
    results['Embedding (rubert-tiny2)'] = {'accuracy': emb_acc_l1, 'f1_macro': emb_f1_l1}

if HAS_SETFIT and 'setfit_acc' in dir():
    results['SetFit'] = {'accuracy': setfit_acc, 'f1_macro': setfit_f1}

if 'cascade_acc' in dir():
    results['Каскад'] = {'accuracy': cascade_acc, 'f1_macro': cascade_f1}

# Создание DataFrame
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)
print(results_df)

# Сохранение
results_df.to_csv(OUTPUT_TABLES / 'v5_methods_comparison.csv')

# Лучшая модель
best_model_name = results_df['f1_macro'].idxmax()
best_f1 = results_df.loc[best_model_name, 'f1_macro']
print(f'\nЛучшая модель: {best_model_name} (F1 Macro: {best_f1:.4f})')

=== Сводка результатов (legacy) ===
                          accuracy  f1_macro
TF-IDF + LR (L1)            0.9634    0.9654
TF-IDF + LR (L2)            0.9511    0.9526
GridSearchCV SVC (L1)       0.9828    0.9824
GridSearchCV SVC (L2)       0.9589    0.9600
Embedding (rubert-tiny2)    0.9389    0.9368
Каскад                      0.5533    0.1910

Лучшая модель: GridSearchCV SVC (L1) (F1 Macro: 0.9824)


In [20]:
# =============================================================================
# Ячейка 19: Матрица ошибок (лучшая модель L1, legacy)
# =============================================================================
print('=== Матрица ошибок (GridSearchCV L1) ===')

cm = confusion_matrix(y_val_l1, y_pred_svc_l1, labels=INTENT_LABELS_L1)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=INTENT_LABELS_L1, yticklabels=INTENT_LABELS_L1)
plt.title('Матрица ошибок — L1 (GridSearchCV SVC)')
plt.xlabel('Предсказано')
plt.ylabel('Истинное')
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'v5_confusion_matrix_l1.png', dpi=150)
plt.show()

=== Матрица ошибок (GridSearchCV L1) ===


## Экспорт моделей (legacy v4.0)

In [21]:
# =============================================================================
# Ячейка 20: Сохранение моделей (legacy)
# =============================================================================
print('=== Сохранение моделей ===')

# TF-IDF векторизатор
joblib.dump(tfidf, OUTPUT_MODELS / 'tfidf_vectorizer_v5.joblib')
print(f'Сохранено: tfidf_vectorizer_v5.joblib')

# Лучшая модель L1
joblib.dump(grid_search_l1.best_estimator_, OUTPUT_MODELS / 'l1_gridsearch_best_v5.joblib')
print(f'Сохранено: l1_gridsearch_best_v5.joblib')

# Лучшая модель L2
joblib.dump(grid_search_l2.best_estimator_, OUTPUT_MODELS / 'l2_gridsearch_best_v5.joblib')
print(f'Сохранено: l2_gridsearch_best_v5.joblib')

# Сохранение Embedding классификатора
if HAS_SBERT and 'emb_clf_l1_primary' in dir():
    emb_clf_l1_primary.save(OUTPUT_MODELS / 'l1_embedding_rubert_v5')
    print(f'Сохранено: l1_embedding_rubert_v5')

# Сохранение результатов в JSON
results_json = {
    'version': '5.0',
    'timestamp': datetime.now().isoformat(),
    'dataset': {
        'name': 'v5',
        'total': len(df_full),
        'train': len(df_train),
        'val': len(df_val),
        'test': len(df_test),
        'l1_classes': len(INTENT_LABELS_L1),
        'l2_classes': len(INTENT_LABELS_L2),
    },
    'results': results_df.to_dict(),
    'best_model': best_model_name,
    'best_f1_macro': float(best_f1),
}

with open(OUTPUT_DIR / 'd1_v5_results_legacy.json', 'w') as f:
    json.dump(results_json, f, indent=2, ensure_ascii=False)
print(f'Сохранено: d1_v5_results_legacy.json')

print('\n=== Все модели сохранены ===')

=== Сохранение моделей ===
Сохранено: tfidf_vectorizer_v5.joblib
Сохранено: l1_gridsearch_best_v5.joblib
Сохранено: l2_gridsearch_best_v5.joblib
Сохранено: l1_embedding_rubert_v5
Сохранено: d1_v5_results_legacy.json

=== Все модели сохранены ===


In [22]:
# =============================================================================
# Ячейка 21: Итоговая сводка (legacy)
# =============================================================================
print('=' * 60)
print('D1 Surface Classifier v5.0 — ЭКСПЕРИМЕНТ ЗАВЕРШЁН (legacy)')
print('=' * 60)

print(f'\nДатасет: v5 ({len(df_full):,} сэмплов, {len(INTENT_LABELS_L2)} L2 классов)')
print(f'Лучшая модель: {best_model_name}')
print(f'Лучший F1 Macro: {best_f1:.4f}')

print(f'\nМетрики Feedback:')
if 'neg_recall' in dir():
    print(f'  Полнота негативных: {neg_recall:.4f}')
if 'pos_recall' in dir():
    print(f'  Полнота позитивных: {pos_recall:.4f}')
if 'feedback_f1' in dir():
    print(f'  F1 Feedback: {feedback_f1:.4f}')

if 'llm_reduction' in dir():
    print(f'\nСнижение LLM вызовов: {llm_reduction:.1f}%')

print(f'\nАртефакты: {OUTPUT_DIR}')
print('\n⚠️ Проблемы legacy (см. аудит):')
print('  - Два параллельных классификатора (L1 и L2) вместо одного')
print('  - Test set НЕ используется (оценка только на val)')
print('  - cv=3, нет CI, нет bootstrap')
print('  - Каскад F1=0.261 — broken')
print('\n→ Исправлено в секции v5.0 ниже')

D1 Surface Classifier v5.0 — ЭКСПЕРИМЕНТ ЗАВЕРШЁН (legacy)

Датасет: v5 (12,000 сэмплов, 24 L2 классов)
Лучшая модель: GridSearchCV SVC (L1)
Лучший F1 Macro: 0.9824

Метрики Feedback:
  Полнота негативных: 0.9700
  Полнота позитивных: 0.9833
  F1 Feedback: 0.6060

Снижение LLM вызовов: 100.0%

Артефакты: outputs

⚠️ Проблемы legacy (см. аудит):
  - Два параллельных классификатора (L1 и L2) вместо одного
  - Test set НЕ используется (оценка только на val)
  - cv=3, нет CI, нет bootstrap
  - Каскад F1=0.261 — broken

→ Исправлено в секции v5.0 ниже


---

# D1 v5.0 — Flat Classifier + Statistical Rigor

**Ключевое изменение:** ОДНА модель на L2 (24 класса). L1 получаем агрегацией через `L2_TO_L1`.

**План (7 шагов):**
1. Одна flat модель на L2 → L1 агрегацией
2. Repeated Stratified K-Fold (5×3) + CI 95% + Wilcoxon
3. Held-out Test Set — финальная оценка
4. Robustness Testing (typo noise, truncation, concatenation)
5. Calibration Analysis + reliability diagram
6. Ablation Study (5 параметров)
7. Error Analysis — качественный

**Данные:** v5 (12K сэмплов, 8400 train / 1800 val / 1800 test)
**Ограничение:** Все данные синтетические.

## Шаг 1: Одна flat модель на L2 (24 класса), L1 агрегацией

**Цель:** Обучить ОДНУ модель на L2, получать L1 через `L2_TO_L1`.
Сравнить 3 метода: TF-IDF+SVC, TF-IDF+LR, Embedding+LogReg.

In [23]:
# =============================================================================
# Шаг 1.1: Дополнительные импорты для statistical rigor
# =============================================================================
from scipy import stats
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.pipeline import Pipeline
from collections import Counter

# Маппинг L2 → L1 (из taxonomy.py, уже импортирован)
print(f"L2_TO_L1: {len(L2_TO_L1)} маппингов")
print(f"Пример: symptom → {L2_TO_L1['symptom']}, price → {L2_TO_L1['price']}")

# Вспомогательная функция: L2 предсказания → L1
def derive_l1_from_l2(y_l2):
    """Агрегация L2 предсказаний в L1 через L2_TO_L1 маппинг."""
    return np.array([L2_TO_L1.get(label, "faq") for label in y_l2])

L2_TO_L1: 24 маппингов
Пример: symptom → anamnesis, price → faq


In [24]:
# =============================================================================
# Шаг 1.2: Flat модель — TF-IDF + LinearSVC на L2 (24 класса)
# =============================================================================
# Используем TF-IDF из ячейки 5 (уже обучен на train)
# X_train_tfidf, X_val_tfidf, X_test_tfidf — уже готовы

print("=== Flat Model: TF-IDF + LinearSVC на L2 (24 класса) ===")

# Веса классов L2
class_weights_l2 = get_sklearn_class_weights('l2')

# Одна модель на L2
svc_flat_l2 = LinearSVC(
    C=1.0,
    dual='auto',
    class_weight=class_weights_l2,
    random_state=SEED,
    max_iter=2000,
)
svc_flat_l2.fit(X_train_tfidf, y_train_l2)

# Предсказания L2 на val
y_pred_l2_svc = svc_flat_l2.predict(X_val_tfidf)

# Агрегация L2 → L1
y_pred_l1_derived_svc = derive_l1_from_l2(y_pred_l2_svc)

# Метрики L2
f1_l2_svc = f1_score(y_val_l2, y_pred_l2_svc, average='macro')
acc_l2_svc = accuracy_score(y_val_l2, y_pred_l2_svc)

# Метрики L1 (агрегированные)
f1_l1_svc = f1_score(y_val_l1, y_pred_l1_derived_svc, average='macro')
acc_l1_svc = accuracy_score(y_val_l1, y_pred_l1_derived_svc)

print(f"\n--- TF-IDF + LinearSVC (flat L2 → L1) ---")
print(f"L2: F1-macro={f1_l2_svc:.4f}, Accuracy={acc_l2_svc:.4f}")
print(f"L1: F1-macro={f1_l1_svc:.4f}, Accuracy={acc_l1_svc:.4f}")
print(f"\nL2 Classification Report:")
print(classification_report(y_val_l2, y_pred_l2_svc, zero_division=0))

=== Flat Model: TF-IDF + LinearSVC на L2 (24 класса) ===

--- TF-IDF + LinearSVC (flat L2 → L1) ---
L2: F1-macro=0.9600, Accuracy=0.9589
L1: F1-macro=0.9793, Accuracy=0.9806

L2 Classification Report:
                  precision    recall  f1-score   support

          cancel       0.95      0.99      0.97        75
     clinic_info       0.95      0.96      0.95        75
       complaint       0.99      0.95      0.97        75
    confirmation       0.97      0.93      0.95        75
        farewell       1.00      0.95      0.97        75
        followup       0.96      0.92      0.94        75
       gratitude       0.92      0.95      0.94        76
        greeting       0.96      0.87      0.91        75
negative_general       0.97      0.92      0.95        75
negative_quality       0.99      0.97      0.98        75
negative_service       0.99      0.99      0.99        75
  negative_staff       0.97      1.00      0.99        75
 new_appointment       1.00      0.99      0

In [25]:
# =============================================================================
# Шаг 1.3: Flat модель — TF-IDF + Logistic Regression на L2
# =============================================================================
print("=== Flat Model: TF-IDF + LogisticRegression на L2 (24 класса) ===")

lr_flat_l2 = LogisticRegression(
    max_iter=2000,
    class_weight=class_weights_l2,
    random_state=SEED,
    n_jobs=-1,
    solver='lbfgs',
)
lr_flat_l2.fit(X_train_tfidf, y_train_l2)

# Предсказания L2 на val
y_pred_l2_lr = lr_flat_l2.predict(X_val_tfidf)
y_pred_l1_derived_lr = derive_l1_from_l2(y_pred_l2_lr)

# Метрики
f1_l2_lr = f1_score(y_val_l2, y_pred_l2_lr, average='macro')
acc_l2_lr = accuracy_score(y_val_l2, y_pred_l2_lr)
f1_l1_lr = f1_score(y_val_l1, y_pred_l1_derived_lr, average='macro')
acc_l1_lr = accuracy_score(y_val_l1, y_pred_l1_derived_lr)

print(f"\n--- TF-IDF + LogReg (flat L2 → L1) ---")
print(f"L2: F1-macro={f1_l2_lr:.4f}, Accuracy={acc_l2_lr:.4f}")
print(f"L1: F1-macro={f1_l1_lr:.4f}, Accuracy={acc_l1_lr:.4f}")

=== Flat Model: TF-IDF + LogisticRegression на L2 (24 класса) ===

--- TF-IDF + LogReg (flat L2 → L1) ---
L2: F1-macro=0.9526, Accuracy=0.9511
L1: F1-macro=0.9688, Accuracy=0.9678


In [26]:
# =============================================================================
# Шаг 1.4: Flat модель — Embedding (rubert-tiny2) + LogReg на L2
# =============================================================================
print("=== Flat Model: Embedding (rubert-tiny2) + LogReg на L2 (24 класса) ===")

if HAS_SBERT:
    # Обучаем EmbeddingClassifier на L2 (в v4 был только L1)
    emb_clf_l2 = EmbeddingClassifier(
        model_name='cointegrated/rubert-tiny2',
    )
    
    print("Обучение embedding classifier на L2...")
    emb_clf_l2.fit(X_train.tolist(), y_train_l2.tolist())
    
    # Предсказания L2 на val
    results_emb = emb_clf_l2.predict_with_confidence(X_val.tolist())
    y_pred_l2_emb = np.array([r[0] for r in results_emb])
    y_pred_l1_derived_emb = derive_l1_from_l2(y_pred_l2_emb)
    
    # Метрики
    f1_l2_emb = f1_score(y_val_l2, y_pred_l2_emb, average='macro')
    acc_l2_emb = accuracy_score(y_val_l2, y_pred_l2_emb)
    f1_l1_emb = f1_score(y_val_l1, y_pred_l1_derived_emb, average='macro')
    acc_l1_emb = accuracy_score(y_val_l1, y_pred_l1_derived_emb)
    
    print(f"\n--- Embedding + LogReg (flat L2 → L1) ---")
    print(f"L2: F1-macro={f1_l2_emb:.4f}, Accuracy={acc_l2_emb:.4f}")
    print(f"L1: F1-macro={f1_l1_emb:.4f}, Accuracy={acc_l1_emb:.4f}")
else:
    print("SKIP: sentence-transformers не установлен")
    f1_l2_emb, acc_l2_emb, f1_l1_emb, acc_l1_emb = 0, 0, 0, 0

=== Flat Model: Embedding (rubert-tiny2) + LogReg на L2 (24 класса) ===
Обучение embedding classifier на L2...


Batches:   0%|          | 0/263 [00:00<?, ?it/s]


--- Embedding + LogReg (flat L2 → L1) ---
L2: F1-macro=0.9378, Accuracy=0.9378
L1: F1-macro=0.9649, Accuracy=0.9656


In [27]:
# =============================================================================
# Шаг 1.5: Сводная таблица — flat L2 модели (val set)
# =============================================================================
print("=== Сводка: Flat L2 модели (validation set) ===\n")

flat_results = {
    'TF-IDF + SVC (L2→L1)': {
        'F1 L2': f1_l2_svc, 'Acc L2': acc_l2_svc,
        'F1 L1': f1_l1_svc, 'Acc L1': acc_l1_svc,
    },
    'TF-IDF + LR (L2→L1)': {
        'F1 L2': f1_l2_lr, 'Acc L2': acc_l2_lr,
        'F1 L1': f1_l1_lr, 'Acc L1': acc_l1_lr,
    },
}

if HAS_SBERT and f1_l2_emb > 0:
    flat_results['Embedding + LR (L2→L1)'] = {
        'F1 L2': f1_l2_emb, 'Acc L2': acc_l2_emb,
        'F1 L1': f1_l1_emb, 'Acc L1': acc_l1_emb,
    }

df_flat = pd.DataFrame(flat_results).T.round(4)
print(df_flat)

# Определяем лучшую модель по F1 L2
best_flat_name = df_flat['F1 L2'].idxmax()
best_flat_f1 = df_flat.loc[best_flat_name, 'F1 L2']
print(f"\n✅ Лучшая flat модель (по F1 L2): {best_flat_name}")
print(f"   F1 L2 = {best_flat_f1:.4f}, F1 L1 = {df_flat.loc[best_flat_name, 'F1 L1']:.4f}")
print(f"\n⚠️ Это предварительные результаты на val set.")
print(f"   Финальная оценка — на held-out test set (Шаг 3).")

# Сохраняем таблицу
df_flat.to_csv(OUTPUT_TABLES / 'v5_flat_models_comparison.csv')
print(f"\nСохранено: {OUTPUT_TABLES / 'v5_flat_models_comparison.csv'}")

=== Сводка: Flat L2 модели (validation set) ===

                         F1 L2  Acc L2   F1 L1  Acc L1
TF-IDF + SVC (L2→L1)    0.9600  0.9589  0.9793  0.9806
TF-IDF + LR (L2→L1)     0.9526  0.9511  0.9688  0.9678
Embedding + LR (L2→L1)  0.9378  0.9378  0.9649  0.9656

✅ Лучшая flat модель (по F1 L2): TF-IDF + SVC (L2→L1)
   F1 L2 = 0.9600, F1 L1 = 0.9793

⚠️ Это предварительные результаты на val set.
   Финальная оценка — на held-out test set (Шаг 3).

Сохранено: outputs/tables/v5_flat_models_comparison.csv


## Шаг 2: Proper Statistical Protocol — Repeated Stratified K-Fold

**Протокол:** 5 folds × 3 repeats = 15 runs для каждого метода.
- 95% CI через t-распределение
- Wilcoxon signed-rank test для попарного сравнения методов
- Данные: train+val (10200 сэмплов), test (1800) — НЕ трогаем

In [28]:
# =============================================================================
# Шаг 2.1: Repeated Stratified K-Fold (5x3 = 15 runs)
# =============================================================================
# Объединяем train + val для CV (test set НЕ трогаем)
X_cv = np.concatenate([X_train, X_val])
y_cv_l2 = np.concatenate([y_train_l2, y_val_l2])
y_cv_l1 = np.concatenate([y_train_l1, y_val_l1])

print(f"CV данные: {len(X_cv)} сэмплов (train {len(X_train)} + val {len(X_val)})")
print(f"Test set: {len(X_test)} сэмплов (НЕ используется в CV)")

N_SPLITS = 5
N_REPEATS = 3

rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)

cv_scores = {
    'tfidf_svc': {'f1_l2': [], 'f1_l1': []},
    'tfidf_lr':  {'f1_l2': [], 'f1_l1': []},
    'embedding': {'f1_l2': [], 'f1_l1': []},
}

# --- Pre-compute embeddings (frozen model, no data leakage) ---
if HAS_SBERT:
    print("\nПредвычисление embeddings (rubert-tiny2)...")
    from sentence_transformers import SentenceTransformer
    _sbert_model = SentenceTransformer('cointegrated/rubert-tiny2')
    X_cv_embeddings = _sbert_model.encode(X_cv.tolist(), show_progress_bar=True, batch_size=128)
    print(f"Embeddings shape: {X_cv_embeddings.shape}")
else:
    X_cv_embeddings = None
    print("\nEmbedding пропущен (sentence-transformers недоступен)")

print(f"\nЗапуск Repeated Stratified K-Fold ({N_SPLITS}x{N_REPEATS} = {N_SPLITS*N_REPEATS} runs)...")

for fold_idx, (train_idx, val_idx) in enumerate(tqdm(
        rskf.split(X_cv, y_cv_l2), total=N_SPLITS*N_REPEATS, desc="CV")):
    X_fold_train, X_fold_val = X_cv[train_idx], X_cv[val_idx]
    y_fold_train_l2 = y_cv_l2[train_idx]
    y_fold_val_l2 = y_cv_l2[val_idx]
    y_fold_val_l1 = y_cv_l1[val_idx]
    
    # TF-IDF vectorization (inside fold to avoid data leakage)
    tfidf_cv = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2),
        min_df=2, max_df=0.95, sublinear_tf=True,
    )
    X_tr_tfidf = tfidf_cv.fit_transform(X_fold_train)
    X_vl_tfidf = tfidf_cv.transform(X_fold_val)
    
    # --- Method 1: TF-IDF + SVC ---
    svc = LinearSVC(C=1.0, dual='auto', class_weight=class_weights_l2,
                    random_state=SEED, max_iter=2000)
    svc.fit(X_tr_tfidf, y_fold_train_l2)
    y_pred_svc = svc.predict(X_vl_tfidf)
    
    cv_scores['tfidf_svc']['f1_l2'].append(
        f1_score(y_fold_val_l2, y_pred_svc, average='macro'))
    y_pred_svc_l1 = [L2_TO_L1[l] for l in y_pred_svc]
    cv_scores['tfidf_svc']['f1_l1'].append(
        f1_score(y_fold_val_l1, y_pred_svc_l1, average='macro'))
    
    # --- Method 2: TF-IDF + LR ---
    lr = LogisticRegression(max_iter=1000, class_weight=class_weights_l2,
                            random_state=SEED, n_jobs=-1)
    lr.fit(X_tr_tfidf, y_fold_train_l2)
    y_pred_lr = lr.predict(X_vl_tfidf)
    
    cv_scores['tfidf_lr']['f1_l2'].append(
        f1_score(y_fold_val_l2, y_pred_lr, average='macro'))
    y_pred_lr_l1 = [L2_TO_L1[l] for l in y_pred_lr]
    cv_scores['tfidf_lr']['f1_l1'].append(
        f1_score(y_fold_val_l1, y_pred_lr_l1, average='macro'))
    
    # --- Method 3: Embedding (frozen rubert-tiny2) + LogReg ---
    if X_cv_embeddings is not None:
        X_emb_tr = X_cv_embeddings[train_idx]
        X_emb_vl = X_cv_embeddings[val_idx]
        
        lr_emb = LogisticRegression(max_iter=1000, class_weight=class_weights_l2,
                                    random_state=SEED, n_jobs=-1)
        lr_emb.fit(X_emb_tr, y_fold_train_l2)
        y_pred_emb = lr_emb.predict(X_emb_vl)
        
        cv_scores['embedding']['f1_l2'].append(
            f1_score(y_fold_val_l2, y_pred_emb, average='macro'))
        y_pred_emb_l1 = [L2_TO_L1[l] for l in y_pred_emb]
        cv_scores['embedding']['f1_l1'].append(
            f1_score(y_fold_val_l1, y_pred_emb_l1, average='macro'))

print("\nCV завершен.")

CV данные: 10200 сэмплов (train 8399 + val 1801)
Test set: 1800 сэмплов (НЕ используется в CV)

Предвычисление embeddings (rubert-tiny2)...


Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Embeddings shape: (10200, 312)

Запуск Repeated Stratified K-Fold (5x3 = 15 runs)...


CV: 100%|██████████| 15/15 [00:39<00:00,  2.62s/it]


CV завершен.


In [29]:
# =============================================================================
# Шаг 2.2: CI 95% + Wilcoxon signed-rank test (3 метода)
# =============================================================================
print("=== Statistical Analysis: CI 95% + Wilcoxon ===\n")

def compute_ci_95(scores):
    """Вычисление 95% доверительного интервала через t-распределение."""
    n = len(scores)
    mean = np.mean(scores)
    se = stats.sem(scores)
    ci_lo, ci_hi = stats.t.interval(0.95, df=n-1, loc=mean, scale=se)
    return mean, ci_lo, ci_hi

# --- Таблица результатов с CI ---
print(f"{'Метод':25s}| {'F1 L2 mean [CI 95%]':28s} | {'F1 L1 mean [CI 95%]':28s}")
print("-" * 90)

ci_results = {}
for method, scores in cv_scores.items():
    if not scores['f1_l2']:
        continue
    mean_l2, lo_l2, hi_l2 = compute_ci_95(scores['f1_l2'])
    mean_l1, lo_l1, hi_l1 = compute_ci_95(scores['f1_l1'])
    ci_results[method] = {
        'f1_l2_mean': mean_l2, 'f1_l2_ci': (lo_l2, hi_l2),
        'f1_l1_mean': mean_l1, 'f1_l1_ci': (lo_l1, hi_l1),
    }
    print(f"{method:25s}| {mean_l2:.4f} [{lo_l2:.4f}, {hi_l2:.4f}] | {mean_l1:.4f} [{lo_l1:.4f}, {hi_l1:.4f}]")

# --- Pairwise Wilcoxon signed-rank tests ---
print("\n--- Pairwise Wilcoxon signed-rank tests (F1 L2) ---")

methods_with_scores = [m for m in cv_scores if cv_scores[m]['f1_l2']]
n_pairs = len(methods_with_scores) * (len(methods_with_scores) - 1) // 2
ALPHA = 0.05
alpha_corrected = ALPHA / max(n_pairs, 1)
print(f"Bonferroni correction: alpha={ALPHA} / {n_pairs} pairs = {alpha_corrected:.4f}\n")

for i, m1 in enumerate(methods_with_scores):
    for m2 in methods_with_scores[i+1:]:
        stat_l2, p_l2 = stats.wilcoxon(cv_scores[m1]['f1_l2'], cv_scores[m2]['f1_l2'])
        effect = np.mean(cv_scores[m1]['f1_l2']) - np.mean(cv_scores[m2]['f1_l2'])
        sig = "***" if p_l2 < alpha_corrected else "n.s."
        winner = m1 if effect > 0 else m2
        print(f"  {m1} vs {m2}: p={p_l2:.6f}, effect={effect:+.4f} [{sig}] → {winner}")

# Определяем лучший метод
best_method = max(ci_results, key=lambda m: ci_results[m]['f1_l2_mean'])
print(f"\n=> Лучший метод: {best_method} (F1 L2 = {ci_results[best_method]['f1_l2_mean']:.4f})")

=== Statistical Analysis: CI 95% + Wilcoxon ===

Метод                    | F1 L2 mean [CI 95%]          | F1 L1 mean [CI 95%]         
------------------------------------------------------------------------------------------
tfidf_svc                | 0.9638 [0.9616, 0.9661] | 0.9789 [0.9771, 0.9806]
tfidf_lr                 | 0.9567 [0.9550, 0.9584] | 0.9676 [0.9654, 0.9698]
embedding                | 0.8597 [0.8563, 0.8631] | 0.9250 [0.9214, 0.9286]

--- Pairwise Wilcoxon signed-rank tests (F1 L2) ---
Bonferroni correction: alpha=0.05 / 3 pairs = 0.0167

  tfidf_svc vs tfidf_lr: p=0.000061, effect=+0.0072 [***] → tfidf_svc
  tfidf_svc vs embedding: p=0.000061, effect=+0.1041 [***] → tfidf_svc
  tfidf_lr vs embedding: p=0.000061, effect=+0.0969 [***] → tfidf_lr

=> Лучший метод: tfidf_svc (F1 L2 = 0.9638)


In [30]:
# =============================================================================
# Шаг 2.3: Визуализация CV результатов (boxplot + CI) — 3 метода
# =============================================================================
methods_with_scores = [m for m in cv_scores if cv_scores[m]['f1_l2']]
labels_map = {
    'tfidf_svc': 'TF-IDF + SVC',
    'tfidf_lr': 'TF-IDF + LR',
    'embedding': 'Embedding + LR',
}
colors_map = {
    'tfidf_svc': 'lightblue',
    'tfidf_lr': 'lightyellow',
    'embedding': 'lightgreen',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Boxplot F1 L2 ---
data_l2 = [cv_scores[m]['f1_l2'] for m in methods_with_scores]
labels = [labels_map.get(m, m) for m in methods_with_scores]
colors = [colors_map.get(m, 'lightgray') for m in methods_with_scores]

bp1 = axes[0].boxplot(data_l2, labels=labels, patch_artist=True)
for patch, color in zip(bp1['boxes'], colors):
    patch.set_facecolor(color)
axes[0].set_title('F1-macro L2 (24 класса) — Repeated 5x3 CV')
axes[0].set_ylabel('F1-macro')
axes[0].axhline(y=0.90, color='green', linestyle='--', alpha=0.7, label='Target (0.90)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# --- Boxplot F1 L1 ---
data_l1 = [cv_scores[m]['f1_l1'] for m in methods_with_scores]

bp2 = axes[1].boxplot(data_l1, labels=labels, patch_artist=True)
for patch, color in zip(bp2['boxes'], colors):
    patch.set_facecolor(color)
axes[1].set_title('F1-macro L1 (5 классов, агрегация) — Repeated 5x3 CV')
axes[1].set_ylabel('F1-macro')
axes[1].axhline(y=0.95, color='green', linestyle='--', alpha=0.7, label='Target (0.95)')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle(f'Repeated Stratified K-Fold ({N_SPLITS}x{N_REPEATS}), n={len(X_cv)}, {len(methods_with_scores)} methods', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'v5_cv_boxplot_f1.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Сохранено: {OUTPUT_FIGURES / 'v5_cv_boxplot_f1.png'}")

Сохранено: outputs/figures/v5_cv_boxplot_f1.png


## Шаг 2.4: Robustness-Augmented CV — typo noise в тренировочных данных

**Гипотеза:** Добавление 10% typo-augmented копий в train повысит робастность TF-IDF модели.

**Протокол:**
1. Для каждого fold добавляем typo-augmented копии (noise_rate=0.05) к train
2. Сравниваем с базовым CV на чистых данных
3. Тестируем оба варианта на чистых и зашумлённых данных

**Embedding гипотеза:** Subword tokenizer (rubert-tiny2) должен быть устойчивее к опечаткам.

In [31]:
# =============================================================================
# Шаг 2.4: Robustness-Augmented CV — TF-IDF + Embedding с typo noise
# =============================================================================

# Функция inject_typo_noise (определена здесь, т.к. в cell 49 она будет позже)
def inject_typo_noise(text, noise_rate=0.1):
    """Замена случайных символов на соседние по клавиатуре (имитация опечаток)."""
    if noise_rate <= 0:
        return text
    ru_neighbors = {
        'а': 'свп', 'б': 'юьд', 'в': 'асыу', 'г': 'нршт', 'д': 'лжэб',
        'е': 'нкау', 'ж': 'дэхз', 'з': 'жхщ', 'и': 'тмш', 'й': 'цфы',
        'к': 'еугн', 'л': 'доргж', 'м': 'исчт', 'н': 'гекр', 'о': 'лдщр',
        'п': 'аер', 'р': 'пногк', 'с': 'вачм', 'т': 'имьг', 'у': 'квцг',
        'ф': 'йыя', 'х': 'зжъщ', 'ц': 'йуф', 'ч': 'ясм', 'ш': 'гищ',
        'щ': 'шзох', 'ъ': 'хэ', 'ы': 'вфйа', 'ь': 'тбо', 'э': 'джъ',
        'ю': 'бь', 'я': 'фчс',
    }
    chars = list(text.lower())
    n_replace = max(1, int(len(chars) * noise_rate))
    indices = random.sample(range(len(chars)), min(n_replace, len(chars)))
    for idx in indices:
        ch = chars[idx]
        if ch in ru_neighbors:
            chars[idx] = random.choice(ru_neighbors[ch])
    return ''.join(chars)

TYPO_NOISE_RATE = 0.05  # 5% символов — реалистичный уровень опечаток
TYPO_AUG_RATIO = 0.10   # 10% train дополняется typo-копиями

print(f"=== Robustness-Augmented CV ===")
print(f"Typo noise rate: {TYPO_NOISE_RATE}")
print(f"Augmented ratio: {TYPO_AUG_RATIO} (10% train gets typo copies)")
print(f"CV: {N_SPLITS}x{N_REPEATS} = {N_SPLITS*N_REPEATS} runs\n")

cv_scores_aug = {
    'tfidf_svc_aug':  {'f1_l2': [], 'f1_l1': []},
    'embedding_aug':  {'f1_l2': [], 'f1_l1': []},
}

for fold_idx, (train_idx, val_idx) in enumerate(tqdm(
        rskf.split(X_cv, y_cv_l2), total=N_SPLITS*N_REPEATS, desc="Aug CV")):
    X_fold_train, X_fold_val = X_cv[train_idx], X_cv[val_idx]
    y_fold_train_l2 = y_cv_l2[train_idx]
    y_fold_val_l2 = y_cv_l2[val_idx]
    y_fold_val_l1 = y_cv_l1[val_idx]
    
    # --- Typo augmentation: добавляем 10% зашумлённых копий ---
    n_aug = int(len(X_fold_train) * TYPO_AUG_RATIO)
    aug_indices = np.random.choice(len(X_fold_train), n_aug, replace=False)
    X_aug = np.array([inject_typo_noise(X_fold_train[i], TYPO_NOISE_RATE) for i in aug_indices])
    y_aug = y_fold_train_l2[aug_indices]
    
    X_train_augmented = np.concatenate([X_fold_train, X_aug])
    y_train_augmented = np.concatenate([y_fold_train_l2, y_aug])
    
    # --- TF-IDF + SVC (augmented) ---
    tfidf_aug = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2),
        min_df=2, max_df=0.95, sublinear_tf=True,
    )
    X_tr_tfidf_aug = tfidf_aug.fit_transform(X_train_augmented)
    X_vl_tfidf_aug = tfidf_aug.transform(X_fold_val)
    
    svc_aug = LinearSVC(C=1.0, dual='auto', class_weight=class_weights_l2,
                        random_state=SEED, max_iter=2000)
    svc_aug.fit(X_tr_tfidf_aug, y_train_augmented)
    y_pred_aug = svc_aug.predict(X_vl_tfidf_aug)
    
    cv_scores_aug['tfidf_svc_aug']['f1_l2'].append(
        f1_score(y_fold_val_l2, y_pred_aug, average='macro'))
    y_pred_aug_l1 = [L2_TO_L1[l] for l in y_pred_aug]
    cv_scores_aug['tfidf_svc_aug']['f1_l1'].append(
        f1_score(y_fold_val_l1, y_pred_aug_l1, average='macro'))
    
    # --- Embedding (augmented) ---
    if X_cv_embeddings is not None:
        # Encode augmented texts
        X_aug_emb = _sbert_model.encode(X_aug.tolist(), show_progress_bar=False, batch_size=128)
        X_emb_tr_aug = np.concatenate([X_cv_embeddings[train_idx], X_aug_emb])
        X_emb_vl = X_cv_embeddings[val_idx]
        
        lr_emb_aug = LogisticRegression(max_iter=1000, class_weight=class_weights_l2,
                                        random_state=SEED, n_jobs=-1)
        lr_emb_aug.fit(X_emb_tr_aug, y_train_augmented)
        y_pred_emb_aug = lr_emb_aug.predict(X_emb_vl)
        
        cv_scores_aug['embedding_aug']['f1_l2'].append(
            f1_score(y_fold_val_l2, y_pred_emb_aug, average='macro'))
        y_pred_emb_aug_l1 = [L2_TO_L1[l] for l in y_pred_emb_aug]
        cv_scores_aug['embedding_aug']['f1_l1'].append(
            f1_score(y_fold_val_l1, y_pred_emb_aug_l1, average='macro'))

# --- Сравнительная таблица: baseline vs augmented ---
print("\n=== Comparison: Baseline vs Typo-Augmented (clean validation) ===\n")
print(f"{'Method':>25s} | {'F1 L2 [CI 95%]':>30s} | {'F1 L1 [CI 95%]':>30s}")
print("-" * 92)

all_methods = {**cv_scores, **cv_scores_aug}
for method, scores in all_methods.items():
    if not scores['f1_l2']:
        continue
    l2_arr = np.array(scores['f1_l2'])
    l1_arr = np.array(scores['f1_l1'])
    l2_mean, l2_lo, l2_hi = l2_arr.mean(), *np.percentile(l2_arr, [2.5, 97.5])
    l1_mean, l1_lo, l1_hi = l1_arr.mean(), *np.percentile(l1_arr, [2.5, 97.5])
    print(f"{method:>25s} | {l2_mean:.4f} [{l2_lo:.4f}, {l2_hi:.4f}] | {l1_mean:.4f} [{l1_lo:.4f}, {l1_hi:.4f}]")

print("\nCV (augmented) завершен.")

=== Robustness-Augmented CV ===
Typo noise rate: 0.05
Augmented ratio: 0.1 (10% train gets typo copies)
CV: 5x3 = 15 runs



Aug CV: 100%|██████████| 15/15 [00:40<00:00,  2.67s/it]


=== Comparison: Baseline vs Typo-Augmented (clean validation) ===

                   Method |                 F1 L2 [CI 95%] |                 F1 L1 [CI 95%]
--------------------------------------------------------------------------------------------
                tfidf_svc | 0.9638 [0.9580, 0.9701] | 0.9789 [0.9733, 0.9842]
                 tfidf_lr | 0.9567 [0.9523, 0.9617] | 0.9676 [0.9607, 0.9730]
                embedding | 0.8597 [0.8513, 0.8705] | 0.9250 [0.9126, 0.9347]
            tfidf_svc_aug | 0.9648 [0.9577, 0.9700] | 0.9801 [0.9733, 0.9853]
            embedding_aug | 0.8627 [0.8533, 0.8736] | 0.9272 [0.9140, 0.9373]

CV (augmented) завершен.


## Шаг 3: Held-out Test Set — финальная оценка

**Протокол:** Лучшая модель из CV обучается на ПОЛНОМ train+val, оценивается на test ОДИН РАЗ.
Это ФИНАЛЬНАЯ метрика для ВКРС. Test set (1800 сэмплов) до этого момента НЕ использовался.

In [32]:
# =============================================================================
# Шаг 3.1: Обучение лучшей модели на ПОЛНОМ train+val, оценка на test
# =============================================================================
print("=== Held-out Test Set — финальная оценка ===\n")

# Лучшая модель по CV: выбираем по f1_l2_mean
best_method = max(ci_results, key=lambda m: ci_results[m]['f1_l2_mean'])
print(f"Лучшая модель по CV (F1 L2): {best_method}")
print(f"  CV F1 L2: {ci_results[best_method]['f1_l2_mean']:.4f} "
      f"[{ci_results[best_method]['f1_l2_ci'][0]:.4f}, "
      f"{ci_results[best_method]['f1_l2_ci'][1]:.4f}]")

# TF-IDF на полном train+val
tfidf_final = TfidfVectorizer(
    max_features=5000, ngram_range=(1, 2),
    min_df=2, max_df=0.95, sublinear_tf=True,
)
X_trainval_tfidf = tfidf_final.fit_transform(X_cv)
X_test_tfidf_final = tfidf_final.transform(X_test)

# Обучение финальной модели на train+val
if best_method == 'tfidf_svc':
    final_model = LinearSVC(
        C=1.0, dual='auto', class_weight=class_weights_l2,
        random_state=SEED, max_iter=2000,
    )
else:
    final_model = LogisticRegression(
        max_iter=2000, class_weight=class_weights_l2,
        random_state=SEED, n_jobs=-1, solver='lbfgs',
    )

final_model.fit(X_trainval_tfidf, y_cv_l2)

# --- ФИНАЛЬНАЯ ОЦЕНКА на test set (ОДИН РАЗ) ---
y_pred_test_l2 = final_model.predict(X_test_tfidf_final)
y_pred_test_l1 = derive_l1_from_l2(y_pred_test_l2)

f1_test_l2 = f1_score(y_test_l2, y_pred_test_l2, average='macro')
acc_test_l2 = accuracy_score(y_test_l2, y_pred_test_l2)
f1_test_l1 = f1_score(y_test_l1, y_pred_test_l1, average='macro')
acc_test_l1 = accuracy_score(y_test_l1, y_pred_test_l1)

print(f"\n{'='*60}")
print(f"  ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ НА TEST SET (n={len(X_test)})")
print(f"{'='*60}")
print(f"  L2 (24 класса): F1-macro = {f1_test_l2:.4f}, Accuracy = {acc_test_l2:.4f}")
print(f"  L1 (5 классов): F1-macro = {f1_test_l1:.4f}, Accuracy = {acc_test_l1:.4f}")
print(f"{'='*60}")
print(f"\n  Target L2 >= 0.90: {'PASS' if f1_test_l2 >= 0.90 else 'FAIL'}")
print(f"  Target L1 >= 0.95: {'PASS' if f1_test_l1 >= 0.95 else 'FAIL'}")

=== Held-out Test Set — финальная оценка ===

Лучшая модель по CV (F1 L2): tfidf_svc
  CV F1 L2: 0.9638 [0.9616, 0.9661]

  ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ НА TEST SET (n=1800)
  L2 (24 класса): F1-macro = 0.9656, Accuracy = 0.9650
  L1 (5 классов): F1-macro = 0.9795, Accuracy = 0.9811

  Target L2 >= 0.90: PASS
  Target L1 >= 0.95: PASS


In [33]:
# =============================================================================
# Шаг 3.2: Confusion Matrix L2 (24 класса) на test set
# =============================================================================
print("=== Confusion Matrix L2 (test set) ===")

cm_l2 = confusion_matrix(y_test_l2, y_pred_test_l2, labels=INTENT_LABELS_L2)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cm_l2, annot=True, fmt='d', cmap='Blues',
    xticklabels=INTENT_LABELS_L2,
    yticklabels=INTENT_LABELS_L2,
    ax=ax,
)
ax.set_title(f'Confusion Matrix L2 (24 класса) — Test Set (n={len(X_test)})\n'
             f'F1-macro={f1_test_l2:.4f}, Accuracy={acc_test_l2:.4f}',
             fontsize=13)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'v5_confusion_matrix_l2_test.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Сохранено: {OUTPUT_FIGURES / 'v5_confusion_matrix_l2_test.png'}")

# Classification report L2
print("\nClassification Report L2 (test set):")
print(classification_report(y_test_l2, y_pred_test_l2, zero_division=0))

=== Confusion Matrix L2 (test set) ===
Сохранено: outputs/figures/v5_confusion_matrix_l2_test.png

Classification Report L2 (test set):
                  precision    recall  f1-score   support

          cancel       1.00      0.99      0.99        75
     clinic_info       0.97      0.96      0.97        75
       complaint       0.96      0.97      0.97        75
    confirmation       0.77      1.00      0.87        75
        farewell       0.99      0.97      0.98        75
        followup       0.96      0.93      0.95        75
       gratitude       0.95      0.96      0.95        75
        greeting       0.97      0.96      0.97        75
negative_general       0.97      0.99      0.98        75
negative_quality       0.97      0.95      0.96        75
negative_service       1.00      0.96      0.98        75
  negative_staff       0.99      0.99      0.99        75
 new_appointment       0.96      0.95      0.95        75
positive_general       1.00      0.97      0.99    

## Шаг 4: Robustness Testing — компенсация отсутствия реальных данных

**Цель:** Оценить degradation curve при моделировании domain shift.
Три типа шума:
1. **Typo noise** — случайные замены/перестановки символов (имитация опечаток)
2. **Truncation** — обрезка текста (имитация коротких сообщений)
3. **Concatenation** — склейка двух сообщений (имитация multi-intent)

**Научная ценность:** degradation curve — ЕДИНСТВЕННЫЙ способ оценить generalization gap на синтетических данных.

In [34]:
# =============================================================================
# Шаг 4.1: Функции генерации шума
# =============================================================================
import string

def inject_typo_noise(text, noise_rate=0.1):
    """Замена случайных символов на соседние по клавиатуре (имитация опечаток).
    
    Args:
        text: исходный текст
        noise_rate: доля символов для замены (0.0 - 1.0)
    Returns:
        текст с опечатками
    """
    if noise_rate <= 0:
        return text
    # Карта соседних клавиш (упрощённая, кириллица)
    ru_neighbors = {
        'а': 'свп', 'б': 'юьд', 'в': 'асыу', 'г': 'нршт', 'д': 'лжэб',
        'е': 'нкау', 'ж': 'дэхз', 'з': 'жхщ', 'и': 'тмш', 'й': 'цфы',
        'к': 'еугн', 'л': 'доргж', 'м': 'исчт', 'н': 'гекр', 'о': 'лдщр',
        'п': 'аер', 'р': 'пногк', 'с': 'вачм', 'т': 'имьг', 'у': 'квцг',
        'ф': 'йыя', 'х': 'зжъщ', 'ц': 'йуф', 'ч': 'ясм', 'ш': 'гищ',
        'щ': 'шзох', 'ъ': 'хэ', 'ы': 'вфйа', 'ь': 'тбо', 'э': 'джъ',
        'ю': 'бь', 'я': 'фчс',
    }
    chars = list(text.lower())
    n_replace = max(1, int(len(chars) * noise_rate))
    indices = random.sample(range(len(chars)), min(n_replace, len(chars)))
    for idx in indices:
        ch = chars[idx]
        if ch in ru_neighbors:
            chars[idx] = random.choice(ru_neighbors[ch])
    return ''.join(chars)


def inject_truncation(text, keep_ratio=0.5):
    """Обрезка текста до заданной доли слов.
    
    Args:
        text: исходный текст
        keep_ratio: доля слов для сохранения (0.0 - 1.0)
    Returns:
        обрезанный текст
    """
    words = text.split()
    n_keep = max(1, int(len(words) * keep_ratio))
    return ' '.join(words[:n_keep])


def inject_concatenation(text, other_texts, n_concat=1):
    """Склейка текста с другим случайным сообщением (multi-intent).
    
    Args:
        text: исходный текст
        other_texts: массив текстов для склейки
        n_concat: количество дополнительных текстов
    Returns:
        склеенный текст
    """
    others = random.sample(list(other_texts), min(n_concat, len(other_texts)))
    return text + ' ' + ' '.join(others)


# Тест функций
sample = "Здравствуйте, у меня болит зуб уже три дня"
print(f"Оригинал:      {sample}")
print(f"Typo (0.1):    {inject_typo_noise(sample, 0.1)}")
print(f"Typo (0.2):    {inject_typo_noise(sample, 0.2)}")
print(f"Truncation 50%: {inject_truncation(sample, 0.5)}")
print(f"Concat:         {inject_concatenation(sample, X_test[:5], 1)[:80]}...")

Оригинал:      Здравствуйте, у меня болит зуб уже три дня
Typo (0.1):    здрввствуйте, у иеня болит зуб уже гри днс
Typo (0.2):    зжрссствуйту, у меня болии згб уже три дня
Truncation 50%: Здравствуйте, у меня болит
Concat:         Здравствуйте, у меня болит зуб уже три дня Привет, можно на другой день?...


In [35]:
# =============================================================================
# Шаг 4.2: Degradation Curve — F1 vs noise_rate
# =============================================================================
print("=== Degradation Curve: F1 vs noise level ===\n")

NOISE_LEVELS = [0.0, 0.05, 0.10, 0.15, 0.20, 0.30]
N_NOISE_SEEDS = 5  # несколько seed для усреднения стохастического шума

degradation = {
    'typo':       {lvl: [] for lvl in NOISE_LEVELS},
    'truncation': {lvl: [] for lvl in NOISE_LEVELS},
}

for seed_i in range(N_NOISE_SEEDS):
    random.seed(SEED + seed_i)
    
    for noise_lvl in NOISE_LEVELS:
        # --- Typo noise ---
        X_noisy_typo = [inject_typo_noise(t, noise_lvl) for t in X_test]
        X_noisy_tfidf = tfidf_final.transform(X_noisy_typo)
        y_pred_noisy = final_model.predict(X_noisy_tfidf)
        f1_noisy = f1_score(y_test_l2, y_pred_noisy, average='macro')
        degradation['typo'][noise_lvl].append(f1_noisy)
        
        # --- Truncation ---
        keep_ratio = 1.0 - noise_lvl  # noise_lvl=0.2 -> keep 80%
        X_trunc = [inject_truncation(t, max(0.3, keep_ratio)) for t in X_test]
        X_trunc_tfidf = tfidf_final.transform(X_trunc)
        y_pred_trunc = final_model.predict(X_trunc_tfidf)
        f1_trunc = f1_score(y_test_l2, y_pred_trunc, average='macro')
        degradation['truncation'][noise_lvl].append(f1_trunc)

# Усреднение и CI
print(f"{'Noise':>8s} | {'Typo F1 [CI]':>28s} | {'Truncation F1 [CI]':>28s}")
print("-" * 72)

deg_summary = {'typo': {}, 'truncation': {}}
for noise_lvl in NOISE_LEVELS:
    for noise_type in ['typo', 'truncation']:
        vals = degradation[noise_type][noise_lvl]
        mean = np.mean(vals)
        if len(vals) > 1:
            lo, hi = np.percentile(vals, [2.5, 97.5])
        else:
            lo, hi = mean, mean
        deg_summary[noise_type][noise_lvl] = (mean, lo, hi)
    
    t_m, t_lo, t_hi = deg_summary['typo'][noise_lvl]
    tr_m, tr_lo, tr_hi = deg_summary['truncation'][noise_lvl]
    print(f"{noise_lvl:>8.2f} | {t_m:.4f} [{t_lo:.4f}, {t_hi:.4f}] | {tr_m:.4f} [{tr_lo:.4f}, {tr_hi:.4f}]")

# Восстанавливаем seed
random.seed(SEED)

=== Degradation Curve: F1 vs noise level ===

   Noise |                 Typo F1 [CI] |           Truncation F1 [CI]
------------------------------------------------------------------------
    0.00 | 0.9656 [0.9656, 0.9656] | 0.9656 [0.9656, 0.9656]
    0.05 | 0.8653 [0.8565, 0.8721] | 0.8933 [0.8933, 0.8933]
    0.10 | 0.7608 [0.7547, 0.7691] | 0.8933 [0.8933, 0.8933]
    0.15 | 0.6357 [0.6283, 0.6455] | 0.8913 [0.8913, 0.8913]
    0.20 | 0.5153 [0.4996, 0.5269] | 0.8711 [0.8711, 0.8711]
    0.30 | 0.3316 [0.3189, 0.3448] | 0.7956 [0.7956, 0.7956]


In [36]:
# =============================================================================
# Шаг 4.3: Визуализация degradation curve
# =============================================================================
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'typo': '#e74c3c', 'truncation': '#3498db'}
labels_map = {'typo': 'Typo noise (замена символов)', 'truncation': 'Truncation (обрезка текста)'}

for noise_type in ['typo', 'truncation']:
    means = [deg_summary[noise_type][lvl][0] for lvl in NOISE_LEVELS]
    los = [deg_summary[noise_type][lvl][1] for lvl in NOISE_LEVELS]
    his = [deg_summary[noise_type][lvl][2] for lvl in NOISE_LEVELS]
    
    ax.plot(NOISE_LEVELS, means, 'o-', color=colors[noise_type],
            label=labels_map[noise_type], linewidth=2, markersize=6)
    ax.fill_between(NOISE_LEVELS, los, his, color=colors[noise_type], alpha=0.15)

# Пороги
ax.axhline(y=0.90, color='green', linestyle='--', alpha=0.7, label='Target F1 L2 (0.90)')
ax.axhline(y=f1_test_l2, color='gray', linestyle=':', alpha=0.5,
           label=f'Baseline F1 (noise=0): {f1_test_l2:.4f}')

ax.set_xlabel('Noise Rate', fontsize=12)
ax.set_ylabel('F1-macro L2', fontsize=12)
ax.set_title('Degradation Curve: F1 vs Noise Level\n'
             '(компенсация отсутствия реальных данных)', fontsize=13)
ax.legend(loc='lower left', fontsize=10)
ax.grid(alpha=0.3)
ax.set_ylim(bottom=max(0, min(los) - 0.05))

plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'v5_degradation_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Сохранено: {OUTPUT_FIGURES / 'v5_degradation_curve.png'}")

# Вывод: при каком уровне шума модель падает ниже порога
for noise_type in ['typo', 'truncation']:
    for lvl in NOISE_LEVELS:
        if deg_summary[noise_type][lvl][0] < 0.90:
            print(f"  {labels_map[noise_type]}: F1 < 0.90 при noise_rate = {lvl}")
            break
    else:
        print(f"  {labels_map[noise_type]}: F1 >= 0.90 при всех уровнях шума")

Сохранено: outputs/figures/v5_degradation_curve.png
  Typo noise (замена символов): F1 < 0.90 при noise_rate = 0.05
  Truncation (обрезка текста): F1 < 0.90 при noise_rate = 0.05


In [37]:
# =============================================================================
# Шаг 4.4: Embedding vs TF-IDF Robustness Comparison (C3)
# =============================================================================
# Гипотеза: subword tokenizer (rubert-tiny2) более устойчив к опечаткам
# чем TF-IDF, который зависит от точного совпадения токенов.
# =============================================================================

if X_cv_embeddings is not None:
    print("=== Embedding vs TF-IDF: Robustness to Typos ===\n")
    
    # Обучаем embedding модель на полном train+val (как final_model для TF-IDF)
    X_cv_emb_all = _sbert_model.encode(
        np.concatenate([X_train, X_val]).tolist(),
        show_progress_bar=True, batch_size=128
    )
    lr_emb_final = LogisticRegression(
        max_iter=1000, class_weight=class_weights_l2,
        random_state=SEED, n_jobs=-1
    )
    lr_emb_final.fit(X_cv_emb_all, np.concatenate([y_train_l2, y_val_l2]))
    
    # Тестируем оба метода на зашумлённых данных
    NOISE_LEVELS_C3 = [0.0, 0.05, 0.10, 0.15, 0.20]
    N_SEEDS_C3 = 5
    
    robustness_cmp = {
        'tfidf_svc': {lvl: [] for lvl in NOISE_LEVELS_C3},
        'embedding':  {lvl: [] for lvl in NOISE_LEVELS_C3},
    }
    
    for seed_i in range(N_SEEDS_C3):
        random.seed(SEED + seed_i)
        for noise_lvl in NOISE_LEVELS_C3:
            X_noisy = [inject_typo_noise(t, noise_lvl) for t in X_test]
            
            # TF-IDF
            X_noisy_tfidf = tfidf_final.transform(X_noisy)
            y_pred_tfidf = final_model.predict(X_noisy_tfidf)
            robustness_cmp['tfidf_svc'][noise_lvl].append(
                f1_score(y_test_l2, y_pred_tfidf, average='macro'))
            
            # Embedding
            X_noisy_emb = _sbert_model.encode(X_noisy, show_progress_bar=False, batch_size=128)
            y_pred_emb = lr_emb_final.predict(X_noisy_emb)
            robustness_cmp['embedding'][noise_lvl].append(
                f1_score(y_test_l2, y_pred_emb, average='macro'))
    
    random.seed(SEED)
    
    # Таблица сравнения
    print(f"{'Noise':>8s} | {'TF-IDF SVC F1 [CI]':>28s} | {'Embedding F1 [CI]':>28s} | {'Δ (Emb-TF)':>10s}")
    print("-" * 82)
    
    for noise_lvl in NOISE_LEVELS_C3:
        tf_vals = np.array(robustness_cmp['tfidf_svc'][noise_lvl])
        emb_vals = np.array(robustness_cmp['embedding'][noise_lvl])
        tf_m, tf_lo, tf_hi = tf_vals.mean(), *np.percentile(tf_vals, [2.5, 97.5])
        emb_m, emb_lo, emb_hi = emb_vals.mean(), *np.percentile(emb_vals, [2.5, 97.5])
        delta = emb_m - tf_m
        print(f"{noise_lvl:>8.2f} | {tf_m:.4f} [{tf_lo:.4f}, {tf_hi:.4f}] | {emb_m:.4f} [{emb_lo:.4f}, {emb_hi:.4f}] | {delta:>+9.4f}")
    
    # Визуализация
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for method, color, marker in [('tfidf_svc', '#e74c3c', 'o'), ('embedding', '#3498db', 's')]:
        means = [np.mean(robustness_cmp[method][lvl]) for lvl in NOISE_LEVELS_C3]
        ci_lo = [np.percentile(robustness_cmp[method][lvl], 2.5) for lvl in NOISE_LEVELS_C3]
        ci_hi = [np.percentile(robustness_cmp[method][lvl], 97.5) for lvl in NOISE_LEVELS_C3]
        
        ax.plot(NOISE_LEVELS_C3, means, f'-{marker}', color=color, label=method, linewidth=2, markersize=8)
        ax.fill_between(NOISE_LEVELS_C3, ci_lo, ci_hi, alpha=0.15, color=color)
    
    ax.set_xlabel('Typo Noise Rate', fontsize=12)
    ax.set_ylabel('F1 Macro (L2)', fontsize=12)
    ax.set_title('Robustness: TF-IDF vs Embedding under Typo Noise', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.5, 1.0)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURES / 'v5_robustness_tfidf_vs_embedding.png', dpi=150)
    plt.show()
    
    # Crossover point
    for lvl in NOISE_LEVELS_C3:
        tf_m = np.mean(robustness_cmp['tfidf_svc'][lvl])
        emb_m = np.mean(robustness_cmp['embedding'][lvl])
        if emb_m > tf_m:
            print(f"\n✓ Embedding превосходит TF-IDF при noise_rate={lvl:.0%} (Emb: {emb_m:.4f} > TF: {tf_m:.4f})")
            break
    else:
        print("\n✗ TF-IDF превосходит Embedding на всех уровнях шума")
else:
    print("Embedding пропущен (sentence-transformers недоступен)")

=== Embedding vs TF-IDF: Robustness to Typos ===



Batches:   0%|          | 0/80 [00:00<?, ?it/s]

   Noise |           TF-IDF SVC F1 [CI] |            Embedding F1 [CI] | Δ (Emb-TF)
----------------------------------------------------------------------------------
    0.00 | 0.9656 [0.9656, 0.9656] | 0.8717 [0.8717, 0.8717] |   -0.0939
    0.05 | 0.8653 [0.8565, 0.8721] | 0.7878 [0.7837, 0.7930] |   -0.0775
    0.10 | 0.7608 [0.7547, 0.7691] | 0.7035 [0.6829, 0.7161] |   -0.0573
    0.15 | 0.6357 [0.6283, 0.6455] | 0.6033 [0.5962, 0.6134] |   -0.0324
    0.20 | 0.5153 [0.4996, 0.5269] | 0.4898 [0.4809, 0.5052] |   -0.0255

✗ TF-IDF превосходит Embedding на всех уровнях шума


## Шаг 5: Calibration Analysis + Reliability Diagram

**Цель:** Проверить, можно ли доверять вероятностям модели для confidence threshold (τ).
LinearSVC не выдаёт вероятности — оборачиваем в `CalibratedClassifierCV(method='sigmoid')`.
Reliability diagram: P(intent)=0.9 → реальная accuracy = ?

In [38]:
# =============================================================================
# Шаг 5.1: Калибровка модели (CalibratedClassifierCV)
# =============================================================================
print("=== Calibration Analysis ===\n")

# LinearSVC не выдаёт вероятности -> оборачиваем в CalibratedClassifierCV
# Обучаем на train+val с внутренней CV для калибровки
cal_model = CalibratedClassifierCV(
    estimator=LinearSVC(
        C=1.0, dual='auto', class_weight=class_weights_l2,
        random_state=SEED, max_iter=2000,
    ),
    cv=5,
    method='sigmoid',
)
cal_model.fit(X_trainval_tfidf, y_cv_l2)

# Вероятности на test set
y_proba_test = cal_model.predict_proba(X_test_tfidf_final)
y_pred_cal = cal_model.predict(X_test_tfidf_final)

# Проверка: калибровка не ухудшила качество
f1_cal = f1_score(y_test_l2, y_pred_cal, average='macro')
print(f"F1-macro L2 (calibrated): {f1_cal:.4f}")
print(f"F1-macro L2 (original):   {f1_test_l2:.4f}")
print(f"Delta: {f1_cal - f1_test_l2:+.4f}")

# Максимальная вероятность для каждого сэмпла
max_proba = y_proba_test.max(axis=1)
print(f"\nDistribution of max confidence:")
print(f"  Mean:   {max_proba.mean():.4f}")
print(f"  Median: {np.median(max_proba):.4f}")
print(f"  P10:    {np.percentile(max_proba, 10):.4f}")
print(f"  P90:    {np.percentile(max_proba, 90):.4f}")

=== Calibration Analysis ===

F1-macro L2 (calibrated): 0.9698
F1-macro L2 (original):   0.9656
Delta: +0.0042

Distribution of max confidence:
  Mean:   0.9304
  Median: 0.9608
  P10:    0.9192
  P90:    0.9706


In [39]:
# =============================================================================
# Шаг 5.2: Reliability Diagram (калибровка вероятностей)
# =============================================================================
print("=== Reliability Diagram ===\n")

# Бинарная задача для reliability diagram: correct/incorrect
y_correct = (y_pred_cal == y_test_l2).astype(int)

# calibration_curve: разбиваем по бинам confidence
fraction_of_positives, mean_predicted_value = calibration_curve(
    y_correct, max_proba, n_bins=10, strategy='uniform'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Reliability diagram ---
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', label='Идеальная калибровка', alpha=0.7)
ax.plot(mean_predicted_value, fraction_of_positives, 's-',
        color='#e74c3c', label='Модель', linewidth=2, markersize=8)
ax.set_xlabel('Средняя предсказанная вероятность', fontsize=11)
ax.set_ylabel('Доля правильных предсказаний', fontsize=11)
ax.set_title('Reliability Diagram\n(калибровка confidence)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# --- Гистограмма confidence ---
ax2 = axes[1]
ax2.hist(max_proba, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax2.axvline(x=0.90, color='red', linestyle='--', label='Threshold 0.90')
ax2.axvline(x=0.70, color='orange', linestyle='--', label='Threshold 0.70')
pct_above_90 = (max_proba >= 0.90).mean() * 100
pct_above_70 = (max_proba >= 0.70).mean() * 100
ax2.set_xlabel('Max Confidence', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title(f'Распределение confidence\n'
              f'{pct_above_90:.1f}% >= 0.90, {pct_above_70:.1f}% >= 0.70', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'v5_reliability_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Сохранено: {OUTPUT_FIGURES / 'v5_reliability_diagram.png'}")

# Accuracy при разных порогах confidence
print("\nAccuracy @ confidence threshold:")
for threshold in [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]:
    mask = max_proba >= threshold
    coverage = mask.mean() * 100
    if mask.sum() > 0:
        acc = accuracy_score(y_test_l2[mask], y_pred_cal[mask])
        print(f"  tau={threshold:.2f}: accuracy={acc:.4f}, coverage={coverage:.1f}%")
    else:
        print(f"  tau={threshold:.2f}: no samples above threshold")

=== Reliability Diagram ===

Сохранено: outputs/figures/v5_reliability_diagram.png

Accuracy @ confidence threshold:
  tau=0.50: accuracy=0.9802, coverage=98.0%
  tau=0.60: accuracy=0.9873, coverage=96.4%
  tau=0.70: accuracy=0.9890, coverage=95.8%
  tau=0.80: accuracy=0.9900, coverage=94.3%
  tau=0.90: accuracy=0.9939, coverage=91.4%
  tau=0.95: accuracy=0.9986, coverage=79.0%


## Шаг 6: Ablation Study — влияние гиперпараметров

**Цель:** Доказать, что выбор гиперпараметров обоснован, а не случаен.
Варьируем 5 параметров TF-IDF/SVC по одному, фиксируя остальные (one-at-a-time ablation).

| Параметр | Базовое значение | Варианты |
|----------|-----------------|----------|
| max_features | 5000 | 1000, 3000, 5000, 10000, 20000 |
| ngram_range | (1,2) | (1,1), (1,2), (1,3), (2,3) |
| C (SVC) | 1.0 | 0.01, 0.1, 1.0, 10.0, 100.0 |
| sublinear_tf | True | True, False |
| min_df | 2 | 1, 2, 5, 10 |

In [40]:
# =============================================================================
# Шаг 6.1: Ablation Study — one-at-a-time
# =============================================================================
print("=== Ablation Study: влияние гиперпараметров на F1 L2 ===\n")

# Базовые параметры (из лучшей модели)
BASE_PARAMS = {
    'max_features': 5000,
    'ngram_range': (1, 2),
    'C': 1.0,
    'sublinear_tf': True,
    'min_df': 2,
}

# Варианты для ablation
ABLATION_GRID = {
    'max_features': [1000, 3000, 5000, 10000, 20000],
    'ngram_range':  [(1, 1), (1, 2), (1, 3), (2, 3)],
    'C':            [0.01, 0.1, 1.0, 10.0, 100.0],
    'sublinear_tf': [True, False],
    'min_df':       [1, 2, 5, 10],
}

def run_ablation_experiment(param_name, param_value, base_params):
    """Запуск одного эксперимента ablation: меняем один параметр, остальные фиксированы.
    
    Используем 3-fold CV на train+val для скорости (не test).
    """
    tfidf_params = {
        'max_features': base_params['max_features'],
        'ngram_range': base_params['ngram_range'],
        'sublinear_tf': base_params['sublinear_tf'],
        'min_df': base_params['min_df'],
        'max_df': 0.95,
    }
    svc_C = base_params['C']
    
    # Подставляем варьируемый параметр
    if param_name == 'C':
        svc_C = param_value
    else:
        tfidf_params[param_name] = param_value
    
    # 3-fold CV (быстрый) на train+val
    skf = RepeatedStratifiedKFold(n_splits=3, n_repeats=1, random_state=SEED)
    scores = []
    
    for tr_idx, vl_idx in skf.split(X_cv, y_cv_l2):
        try:
            vec = TfidfVectorizer(**tfidf_params)
            X_tr = vec.fit_transform(X_cv[tr_idx])
            X_vl = vec.transform(X_cv[vl_idx])
            
            clf = LinearSVC(C=svc_C, dual='auto', class_weight=class_weights_l2,
                           random_state=SEED, max_iter=2000)
            clf.fit(X_tr, y_cv_l2[tr_idx])
            y_pred = clf.predict(X_vl)
            scores.append(f1_score(y_cv_l2[vl_idx], y_pred, average='macro'))
        except Exception as e:
            scores.append(0.0)
    
    return np.mean(scores)

# Запуск ablation
ablation_results = {}

for param_name, values in ABLATION_GRID.items():
    ablation_results[param_name] = {}
    for val in tqdm(values, desc=f"Ablation: {param_name}"):
        f1 = run_ablation_experiment(param_name, val, BASE_PARAMS)
        ablation_results[param_name][str(val)] = f1

# Вывод таблицы
print("\n=== Результаты Ablation Study ===\n")
for param_name, results in ablation_results.items():
    print(f"--- {param_name} ---")
    best_val = max(results, key=results.get)
    for val, f1 in results.items():
        marker = " <-- best" if val == best_val else ""
        base_marker = " (base)" if val == str(BASE_PARAMS[param_name]) else ""
        print(f"  {val:>15s}: F1={f1:.4f}{base_marker}{marker}")
    print()

=== Ablation Study: влияние гиперпараметров на F1 L2 ===



Ablation: min_df: 100%|██████████| 4/4 [00:01<00:00,  2.57it/s]


=== Результаты Ablation Study ===

--- max_features ---
             1000: F1=0.9231
             3000: F1=0.9608 <-- best
             5000: F1=0.9608 (base)
            10000: F1=0.9608
            20000: F1=0.9608

--- ngram_range ---
           (1, 1): F1=0.9566
           (1, 2): F1=0.9608 (base) <-- best
           (1, 3): F1=0.9605
           (2, 3): F1=0.8692

--- C ---
             0.01: F1=0.9146
              0.1: F1=0.9546
              1.0: F1=0.9608 (base) <-- best
             10.0: F1=0.9578
            100.0: F1=0.9562

--- sublinear_tf ---
             True: F1=0.9608 (base)
            False: F1=0.9609 <-- best

--- min_df ---
                1: F1=0.9667 <-- best
                2: F1=0.9608 (base)
                5: F1=0.9377
               10: F1=0.8914



In [41]:
# =============================================================================
# Шаг 6.2: Визуализация ablation — heatmap-style bar charts
# =============================================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (param_name, results) in enumerate(ablation_results.items()):
    ax = axes[idx]
    vals = list(results.keys())
    f1s = list(results.values())
    base_val = str(BASE_PARAMS[param_name])
    
    # Цвета: базовое значение — зелёное, остальные — синие
    colors_bar = ['#2ecc71' if v == base_val else '#3498db' for v in vals]
    
    bars = ax.bar(range(len(vals)), f1s, color=colors_bar, edgecolor='white')
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals, rotation=45, ha='right', fontsize=9)
    ax.set_title(f'{param_name}', fontsize=11, fontweight='bold')
    ax.set_ylabel('F1-macro L2')
    ax.grid(axis='y', alpha=0.3)
    
    # Диапазон Y: чтобы видеть разницу
    if f1s:
        y_min = min(f1s) - 0.01
        y_max = max(f1s) + 0.005
        ax.set_ylim(y_min, y_max)
    
    # Подписи значений на барах
    for bar, f1 in zip(bars, f1s):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
                f'{f1:.4f}', ha='center', va='bottom', fontsize=8)

# Скрыть лишний subplot (6-й)
axes[5].set_visible(False)

plt.suptitle('Ablation Study: влияние каждого гиперпараметра на F1 L2\n'
             '(зелёный = базовое значение)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'v5_ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Сохранено: {OUTPUT_FIGURES / 'v5_ablation_study.png'}")

Сохранено: outputs/figures/v5_ablation_study.png


## Шаг 7: Error Analysis — качественный

**Цель:** Понять, ГДЕ и ПОЧЕМУ модель ошибается.
Категоризация ошибок: Ambiguous / Short text / Cross-class / Rare class.
Для ГЭК качественный error analysis ценнее, чем высокий F1.

In [42]:
# =============================================================================
# Шаг 7.1: Сбор ошибок на test set + Error Rate by Text Length
# =============================================================================
print("=== Error Analysis (test set) ===\n")

# Используем калиброванную модель (есть вероятности)
error_mask = y_pred_cal != y_test_l2

n_errors = error_mask.sum()
n_total = len(y_test_l2)
print(f"Всего ошибок: {n_errors} из {n_total} ({n_errors/n_total*100:.1f}%)")

# Собираем DataFrame ошибок
error_df = pd.DataFrame({
    'text': X_test[error_mask],
    'true_l2': y_test_l2[error_mask],
    'pred_l2': y_pred_cal[error_mask],
    'true_l1': y_test_l1[error_mask],
    'pred_l1': derive_l1_from_l2(y_pred_cal[error_mask]),
    'confidence': max_proba[error_mask],
    'text_len': [len(t.split()) for t in X_test[error_mask]],
})

# --- Категоризация ошибок ---
def categorize_error(row):
    """Категоризация ошибки по типу."""
    # Short text: менее 3 слов (проверяем первым — короткие тексты inherently ambiguous)
    if row['text_len'] <= 3:
        return 'Short text'
    # Low confidence: модель не уверена
    if row['confidence'] < 0.50:
        return 'Low confidence'
    # Cross-class: L1 правильный, L2 неправильный (путаница внутри домена)
    if row['true_l1'] == row['pred_l1']:
        return 'Cross-class (within L1)'
    # Ambiguous: feedback vs conversational (типичная путаница)
    ambiguous_pairs = [
        ('feedback', 'conversational'), ('conversational', 'feedback'),
        ('faq', 'booking'), ('booking', 'faq'),
        ('anamnesis', 'faq'), ('faq', 'anamnesis'),
    ]
    if (row['true_l1'], row['pred_l1']) in ambiguous_pairs:
        return 'Ambiguous (L1 boundary)'
    return 'Other'

error_df['category'] = error_df.apply(categorize_error, axis=1)

# --- Статистика по категориям ---
print("\nРаспределение ошибок по категориям:")
cat_counts = error_df['category'].value_counts()
for cat, count in cat_counts.items():
    pct = count / n_errors * 100
    print(f"  {cat:30s}: {count:4d} ({pct:.1f}%)")

# --- Топ confusion pairs (L2) ---
print("\nТоп-10 пар путаницы (true_l2 -> pred_l2):")
confusion_pairs = error_df.groupby(['true_l2', 'pred_l2']).size().sort_values(ascending=False)
for (true, pred), count in confusion_pairs.head(10).items():
    print(f"  {true:25s} -> {pred:25s}: {count}")

# --- C1: Error Rate by Text Length Bucket (контекстуализация) ---
print("\n\n=== Error Rate by Text Length Bucket ===")
print("(контекстуализация: какой % тестового набора составляют короткие тексты)\n")

all_text_lens = np.array([len(t.split()) for t in X_test])

buckets = [
    ("≤3 words",   all_text_lens <= 3),
    ("4-6 words",  (all_text_lens >= 4) & (all_text_lens <= 6)),
    ("7-10 words", (all_text_lens >= 7) & (all_text_lens <= 10)),
    ("11+ words",  all_text_lens >= 11),
]

print(f"{'Bucket':>15s} | {'Total':>6s} | {'Errors':>6s} | {'Error Rate':>10s} | {'% of Dataset':>12s} | {'% of Errors':>11s}")
print("-" * 75)

for name, mask in buckets:
    n_bucket = mask.sum()
    n_bucket_errors = (error_mask & mask).sum()
    rate = n_bucket_errors / n_bucket * 100 if n_bucket > 0 else 0
    pct_dataset = n_bucket / n_total * 100
    pct_errors = n_bucket_errors / n_errors * 100 if n_errors > 0 else 0
    print(f"{name:>15s} | {n_bucket:>6d} | {n_bucket_errors:>6d} | {rate:>9.1f}% | {pct_dataset:>11.1f}% | {pct_errors:>10.1f}%")

print(f"\nMedian text length (all test): {np.median(all_text_lens):.0f} words")
print(f"Median text length (errors):   {np.median(error_df['text_len']):.0f} words")

# Визуализация error rate по бакетам
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Error rate по бакетам
bucket_names = [b[0] for b in buckets]
bucket_rates = []
bucket_totals = []
for name, mask in buckets:
    n_b = mask.sum()
    n_e = (error_mask & mask).sum()
    bucket_rates.append(n_e / n_b * 100 if n_b > 0 else 0)
    bucket_totals.append(n_b)

bars = axes[0].bar(bucket_names, bucket_rates, color=['#e74c3c', '#f39c12', '#2ecc71', '#3498db'])
axes[0].set_ylabel('Error Rate (%)')
axes[0].set_title('Error Rate by Text Length')
axes[0].axhline(y=n_errors/n_total*100, color='gray', linestyle='--', label=f'Overall: {n_errors/n_total*100:.1f}%')
axes[0].legend()
for bar, rate in zip(bars, bucket_rates):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                f'{rate:.1f}%', ha='center', va='bottom', fontsize=10)

# Распределение длин текстов
axes[1].hist(all_text_lens, bins=20, alpha=0.7, label='All test', color='steelblue')
axes[1].hist(error_df['text_len'], bins=20, alpha=0.7, label='Errors', color='red')
axes[1].set_xlabel('Text Length (words)')
axes[1].set_ylabel('Count')
axes[1].set_title('Text Length Distribution: All vs Errors')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'v5_error_rate_by_length.png', dpi=150)
plt.show()

=== Error Analysis (test set) ===

Всего ошибок: 55 из 1800 (3.1%)

Распределение ошибок по категориям:
  Short text                    :   47 (85.5%)
  Cross-class (within L1)       :    5 (9.1%)
  Other                         :    2 (3.6%)
  Low confidence                :    1 (1.8%)

Топ-10 пар путаницы (true_l2 -> pred_l2):
  clinic_info               -> unclear                  : 3
  followup                  -> unclear                  : 3
  procedure                 -> negative_general         : 2
  complaint                 -> unclear                  : 2
  visit_prep                -> followup                 : 2
  positive_general          -> gratitude                : 2
  services                  -> unclear                  : 2
  reschedule                -> new_appointment          : 2
  price                     -> services                 : 1
  new_appointment           -> services                 : 1


=== Error Rate by Text Length Bucket ===
(контекстуализация: какой

In [43]:
# =============================================================================
# Шаг 7.2: Примеры ошибок по категориям + визуализация
# =============================================================================
print("=== Примеры ошибок по категориям ===\n")

for cat in cat_counts.index[:4]:  # топ-4 категории
    print(f"--- {cat} ---")
    samples = error_df[error_df['category'] == cat].head(3)
    for _, row in samples.iterrows():
        text_short = row['text'][:80] + ('...' if len(row['text']) > 80 else '')
        print(f"  [{row['confidence']:.2f}] {row['true_l2']:>20s} -> {row['pred_l2']:<20s} | {text_short}")
    print()

# --- Визуализация: распределение ошибок ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Ошибки по категориям
ax = axes[0]
cat_counts.plot(kind='barh', ax=ax, color='#e74c3c', edgecolor='white')
ax.set_title('Распределение ошибок по категориям', fontsize=11)
ax.set_xlabel('Количество')
ax.grid(axis='x', alpha=0.3)

# 2. Confidence ошибочных vs правильных
ax2 = axes[1]
ax2.hist(max_proba[~error_mask], bins=30, alpha=0.7, color='#2ecc71', label='Правильные', density=True)
ax2.hist(max_proba[error_mask], bins=30, alpha=0.7, color='#e74c3c', label='Ошибки', density=True)
ax2.set_title('Confidence: правильные vs ошибки', fontsize=11)
ax2.set_xlabel('Max Confidence')
ax2.set_ylabel('Density')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# 3. Длина текста ошибочных vs правильных
ax3 = axes[2]
correct_lens = [len(t.split()) for t in X_test[~error_mask]]
error_lens = error_df['text_len'].values
ax3.hist(correct_lens, bins=20, alpha=0.7, color='#2ecc71', label='Правильные', density=True)
ax3.hist(error_lens, bins=20, alpha=0.7, color='#e74c3c', label='Ошибки', density=True)
ax3.set_title('Длина текста: правильные vs ошибки', fontsize=11)
ax3.set_xlabel('Количество слов')
ax3.set_ylabel('Density')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

plt.suptitle('Error Analysis — качественный анализ ошибок', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'v5_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Сохранено: {OUTPUT_FIGURES / 'v5_error_analysis.png'}")

=== Примеры ошибок по категориям ===

--- Short text ---
  [0.94]             farewell -> greeting             | Привет,п рощайте
  [0.93]            gratitude -> farewell             | Спасибо, дос видания
  [0.10]            complaint -> unclear              | Зубш атается

--- Cross-class (within L1) ---
  [0.83]           reschedule -> new_appointment      | Перенсеите на на следующей неделе
  [0.54]              symptom -> complaint            | Скажите, у меня пульсирующа яболь день
  [0.91]                price -> procedure            | Сколько стоитл ечение каналов?

--- Other ---
  [0.51]     negative_service -> followup             | Так и неп ерезвонили
  [0.59]     negative_quality -> services             | лечение кариеса неп омогло

--- Low confidence ---
  [0.47]      new_appointment -> price                | Записать ребёнка на осмотр

Сохранено: outputs/figures/v5_error_analysis.png


## Итоги D1 v5.0 — Flat Classifier + Statistical Rigor

Все 7 шагов плана реализованы. Сохраняем финальные артефакты и сводку результатов.

In [ ]:
# =============================================================================
# Итоги D1 v5.0: Сохранение артефактов + финальная сводка
# =============================================================================
print("=" * 65)
print("  D1 v5.0 — FLAT CLASSIFIER + STATISTICAL RIGOR — ИТОГИ")
print("=" * 65)

# --- 1. Сохраняем финальную модель ---
joblib.dump(final_model, OUTPUT_MODELS / 'l2_flat_svc_final_v5.joblib')
joblib.dump(tfidf_final, OUTPUT_MODELS / 'tfidf_vectorizer_final_v5.joblib')
joblib.dump(cal_model, OUTPUT_MODELS / 'l2_flat_svc_calibrated_v5.joblib')
print("\nМодели сохранены:")
print(f"  {OUTPUT_MODELS / 'l2_flat_svc_final_v5.joblib'}")
print(f"  {OUTPUT_MODELS / 'tfidf_vectorizer_final_v5.joblib'}")
print(f"  {OUTPUT_MODELS / 'l2_flat_svc_calibrated_v5.joblib'}")

# --- 2. Сохраняем ошибки ---
error_df.to_csv(OUTPUT_TABLES / 'v5_error_analysis.csv', index=False)
print(f"  {OUTPUT_TABLES / 'v5_error_analysis.csv'}")

# --- 3. Сохраняем CV scores (все методы: baseline + augmented) ---
cv_rows = []
all_cv = {**cv_scores}
if 'cv_scores_aug' in dir():
    all_cv.update(cv_scores_aug)
for method, scores in all_cv.items():
    if not scores['f1_l2']:
        continue
    for i in range(len(scores['f1_l2'])):
        cv_rows.append({
            'fold': i,
            'method': method,
            'f1_l2': scores['f1_l2'][i],
            'f1_l1': scores['f1_l1'][i],
        })
cv_scores_df = pd.DataFrame(cv_rows)
cv_scores_df.to_csv(OUTPUT_TABLES / 'v5_cv_scores.csv', index=False)
print(f"  {OUTPUT_TABLES / 'v5_cv_scores.csv'}")

# --- 4. JSON с полными результатами ---

class NumpyEncoder(json.JSONEncoder):
    """JSON encoder для numpy типов (np.float64, np.int64, np.bool_)."""
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.bool_):
            return bool(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

# Augmented CV results (если доступны)
aug_cv_section = {}
for method, scores in (cv_scores_aug if 'cv_scores_aug' in dir() else {}).items():
    if not scores['f1_l2']:
        continue
    arr_l2 = np.array(scores['f1_l2'])
    arr_l1 = np.array(scores['f1_l1'])
    aug_cv_section[method] = {
        'f1_l2_mean': float(arr_l2.mean()),
        'f1_l2_ci_95': [float(np.percentile(arr_l2, 2.5)), float(np.percentile(arr_l2, 97.5))],
        'f1_l1_mean': float(arr_l1.mean()),
        'f1_l1_ci_95': [float(np.percentile(arr_l1, 2.5)), float(np.percentile(arr_l1, 97.5))],
    }

# Robustness comparison (если доступно)
robustness_cmp_section = {}
if 'robustness_cmp' in dir():
    for method in robustness_cmp:
        robustness_cmp_section[method] = {}
        for lvl, vals in robustness_cmp[method].items():
            arr = np.array(vals)
            robustness_cmp_section[method][str(lvl)] = {
                'mean': float(arr.mean()),
                'ci_lo': float(np.percentile(arr, 2.5)),
                'ci_hi': float(np.percentile(arr, 97.5)),
            }

# Error rate by text length
error_by_length = {}
if 'all_text_lens' in dir():
    for name, mask in [("le3", all_text_lens <= 3), ("4to6", (all_text_lens >= 4) & (all_text_lens <= 6)),
                       ("7to10", (all_text_lens >= 7) & (all_text_lens <= 10)), ("11plus", all_text_lens >= 11)]:
        n_b = int(mask.sum())
        n_e = int((error_mask & mask).sum())
        error_by_length[name] = {'total': n_b, 'errors': n_e, 'rate': round(n_e / n_b, 4) if n_b > 0 else 0}

results_v5 = {
    'version': '5.1',
    'timestamp': datetime.now().isoformat(),
    'dataset': {
        'name': 'v5', 'total': len(df_full),
        'train_val': len(X_cv), 'test': len(X_test),
        'l1_classes': len(INTENT_LABELS_L1), 'l2_classes': len(INTENT_LABELS_L2),
    },
    'best_method': best_method,
    'cv_results': {
        method: {
            'f1_l2_mean': ci_results[method]['f1_l2_mean'],
            'f1_l2_ci_95': list(ci_results[method]['f1_l2_ci']),
            'f1_l1_mean': ci_results[method]['f1_l1_mean'],
            'f1_l1_ci_95': list(ci_results[method]['f1_l1_ci']),
        }
        for method in ci_results
    },
    'cv_augmented': aug_cv_section,
    'test_results': {
        'f1_l2': f1_test_l2, 'acc_l2': acc_test_l2,
        'f1_l1': f1_test_l1, 'acc_l1': acc_test_l1,
    },
    'targets': {
        'f1_l2_target': 0.90, 'f1_l2_pass': f1_test_l2 >= 0.90,
        'f1_l1_target': 0.95, 'f1_l1_pass': f1_test_l1 >= 0.95,
    },
    'robustness': {
        noise_type: {
            str(lvl): {'mean': s[0], 'ci_lo': s[1], 'ci_hi': s[2]}
            for lvl, s in summary.items()
        }
        for noise_type, summary in deg_summary.items()
    },
    'robustness_comparison': robustness_cmp_section,
    'calibration': {
        'f1_calibrated': f1_cal,
        'mean_confidence': float(max_proba.mean()),
        'pct_above_090': float((max_proba >= 0.90).mean()),
    },
    'error_analysis': {
        'total_errors': int(n_errors),
        'error_rate': float(n_errors / n_total),
        'categories': cat_counts.to_dict(),
        'by_text_length': error_by_length,
    },
}

with open(OUTPUT_DIR / 'd1_v5_results.json', 'w') as f:
    json.dump(results_v5, f, indent=2, ensure_ascii=False, cls=NumpyEncoder)
print(f"  {OUTPUT_DIR / 'd1_v5_results.json'}")

# --- 5. Финальная сводка ---
print(f"\n{'='*65}")
print(f"  ФИНАЛЬНЫЕ МЕТРИКИ")
print(f"{'='*65}")
print(f"  Метод:     {best_method}")
print(f"  Данные:    {len(X_cv)} train+val, {len(X_test)} test")
methods_with_scores = [m for m in cv_scores if cv_scores[m]['f1_l2']]
print(f"  Методов в CV: {len(methods_with_scores)} ({', '.join(methods_with_scores)})")
print(f"")
print(f"  CV (5x3):  F1 L2 = {ci_results[best_method]['f1_l2_mean']:.4f} "
      f"[{ci_results[best_method]['f1_l2_ci'][0]:.4f}, "
      f"{ci_results[best_method]['f1_l2_ci'][1]:.4f}]")
print(f"             F1 L1 = {ci_results[best_method]['f1_l1_mean']:.4f} "
      f"[{ci_results[best_method]['f1_l1_ci'][0]:.4f}, "
      f"{ci_results[best_method]['f1_l1_ci'][1]:.4f}]")
if aug_cv_section:
    print(f"\n  Augmented CV (typo 5%):")
    for m, s in aug_cv_section.items():
        print(f"    {m}: F1 L2 = {s['f1_l2_mean']:.4f} [{s['f1_l2_ci_95'][0]:.4f}, {s['f1_l2_ci_95'][1]:.4f}]")
print(f"")
print(f"  TEST SET:  F1 L2 = {f1_test_l2:.4f}  {'PASS' if f1_test_l2 >= 0.90 else 'FAIL'} (target >= 0.90)")
print(f"             F1 L1 = {f1_test_l1:.4f}  {'PASS' if f1_test_l1 >= 0.95 else 'FAIL'} (target >= 0.95)")
print(f"")
print(f"  Ошибки:    {n_errors}/{n_total} ({n_errors/n_total*100:.1f}%)")
if error_by_length:
    short = error_by_length.get('le3', {})
    print(f"    ≤3 слов: {short.get('errors',0)}/{short.get('total',0)} "
          f"(error rate {short.get('rate',0)*100:.1f}%, "
          f"dataset share {short.get('total',0)/n_total*100:.1f}%)")
print(f"{'='*65}")
print(f"\n  Артефакты: {OUTPUT_DIR}/")
print(f"  Новые графики:")
print(f"    v5_error_rate_by_length.png")
print(f"    v5_robustness_tfidf_vs_embedding.png")
print(f"\nDone!")

  D1 v5.0 — FLAT CLASSIFIER + STATISTICAL RIGOR — ИТОГИ

Модели сохранены:
  outputs/models/l2_flat_svc_final_v5.joblib
  outputs/models/tfidf_vectorizer_final_v5.joblib
  outputs/models/l2_flat_svc_calibrated_v5.joblib
  outputs/tables/v5_error_analysis.csv
  outputs/tables/v5_cv_scores.csv
  outputs/d1_v5_results.json

  ФИНАЛЬНЫЕ МЕТРИКИ
  Метод:     tfidf_svc
  Данные:    10200 train+val, 1800 test
  Методов в CV: 3 (tfidf_svc, tfidf_lr, embedding)

  CV (5x3):  F1 L2 = 0.9638 [0.9616, 0.9661]
             F1 L1 = 0.9789 [0.9771, 0.9806]

  Augmented CV (typo 5%):
    tfidf_svc_aug: F1 L2 = 0.9648 [0.9577, 0.9700]
    embedding_aug: F1 L2 = 0.8627 [0.8533, 0.8736]

  TEST SET:  F1 L2 = 0.9656  PASS (target >= 0.90)
             F1 L1 = 0.9795  PASS (target >= 0.95)

  Ошибки:    55/1800 (3.1%)
    ≤3 слов: 47/829 (error rate 5.7%, dataset share 46.1%)

  Артефакты: outputs/
  Новые графики:
    v5_error_rate_by_length.png
    v5_robustness_tfidf_vs_embedding.png

Done!


: 